# 🔥 LLM Benchmarking Pipeline for Arduino Projects

## Program-Grounded, Structured RAG Evaluation

This notebook implements an **end-to-end benchmarking pipeline** that evaluates how well Large Language Models can identify hardware components used in Arduino projects.

### Key Idea: Program-Grounded Structured RAG
- **Signal extraction** from raw `.ino` code (includes, API calls, library patterns)
- **Canonical component grounding** via a structured component dictionary
- **Strict JSON output** from LLM with confidence + evidence tagging
- **Metric engine**: Exact-match + relaxed/alias-based evaluation

### Pipeline Stages
1. Data ingestion → 2. Dictionary build → 3. Signal extraction → 4. Prompt construction → 5. LLM call → 6. Normalization → 7. Evaluation → 8. Aggregation → 9. Analysis

In [ ]:
%pwd

In [ ]:
import os

os.chdir('/Users/anonymous_user/Desktop/text-to-arduino')

In [ ]:
# ── CELL 1: IMPORTS ───────────────────────────────────────────────────────────
import os
import re
import json
import logging
import pandas as pd
from typing import List, Dict, Set, Optional, Tuple
from collections import defaultdict
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity
from rank_bm25 import BM25Okapi

load_dotenv(dotenv_path=Path('../.env'))

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

import warnings
from transformers import logging as transformers_logging
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
transformers_logging.set_verbosity_error()

np.random.seed(42)

print('Imports complete')
print(f'NVIDIA_API_KEY_LLAMA33    loaded: {"Yes" if os.getenv("NVIDIA_API_KEY_LLAMA33") else "MISSING"}')
print(f'NVIDIA_API_KEY_PHI3MEDIUM loaded: {"Yes" if os.getenv("NVIDIA_API_KEY_PHI3MEDIUM") else "MISSING"}')
print(f'NVIDIA_API_KEY_PHI3MINI   loaded: {"Yes" if os.getenv("NVIDIA_API_KEY_PHI3MINI") else "MISSING"}')
print(f'NVIDIA_API_KEY_GEMMA      loaded: {"Yes" if os.getenv("NVIDIA_API_KEY_GEMMA") else "MISSING"}')
print(f'NVIDIA_API_KEY_MISTRAL    loaded: {"Yes" if os.getenv("NVIDIA_API_KEY_MISTRAL") else "MISSING"}')

In [ ]:
DATA_DIR   = Path('data')
META_PATH  = DATA_DIR / 'projects_metadata.xlsx'
COMP_PATH  = DATA_DIR / 'components_dictionary.xlsx'
GT_PATH    = DATA_DIR / 'ground_truth_final.json'

In [ ]:
df_meta = pd.read_excel(META_PATH)
df_meta.columns = df_meta.columns.str.strip()

df_comp = pd.read_excel(COMP_PATH)
df_comp.columns = df_comp.columns.str.strip()

print('Project Columns:', df_meta.columns.tolist())
print('Comp Columns:',    df_comp.columns.tolist())

In [ ]:
df_meta = df_meta.rename(columns={
    "Project_ID":  "project_id",
    "Description": "description",
    "Board":       "board",
    "Complexity":  "complexity"
})

meta_lookup = df_meta.set_index("project_id").to_dict("index")

FQBN_MAP = {
    'Arduino Uno':              'arduino:avr:uno',
    'Arduino Uno R4 Wifi Module': 'arduino:renesas_uno:unor4wifi',
    'Arduino Mega':             'arduino:avr:mega',
}

comp_lookup = {
    int(row["Component_Id"]): row.to_dict()
    for _, row in df_comp.iterrows()
}

In [ ]:
with open(GT_PATH) as f:
    projects_raw = json.load(f)

projects = []

for proj in projects_raw:
    pid  = proj.get("project_id")
    meta = meta_lookup.get(pid, {})

    board = meta.get("board", "Arduino Uno")
    fqbn  = FQBN_MAP.get(board, "arduino:avr:uno")

    merged = {
        **proj,
        "description": meta.get("description", proj.get("description", "")),
        "board":       board,
        "board_fqbn":  fqbn,
        "complexity":  meta.get("complexity", "Unknown"),
    }
    projects.append(merged)

print(f"Loaded {len(projects)} merged projects")

# 📦 Task 1: Component Extraction
> **Measures:** Precision, Recall, F1 on predicted vs ground-truth component names

**Method:** LLM-based extraction agent (Module 1) — model reads project description + 
libraries, outputs canonical component names with confidence levels. Evaluated via fuzzy 
name matching with synonym expansion (no component IDs involved).

**Model:** `llama3_3` (meta/llama-3.3-70b-instruct) via NVIDIA API at temperature=0

**Matching:** Token overlap + substring + synonym dictionary + fuzzy sequence match (threshold=0.60)

**GT:** Ground-truth component names derived from comp_lookup, infrastructure excluded

**Status:** ✅ Complete

| Approach | Precision | Recall | F1 |
|---|---|---|---|
| BM25 + Embedding baseline | 0.247 | 0.571 | 0.300 |
| LLM Agent v1 (with IDs) | 0.222 | 0.167 | 0.190 |
| LLM Agent v2 (name-based) | 0.833 | 0.722 | 0.767 |
| LLM Agent v3 (synonyms) | 0.867 | 0.806 | 0.822 |
| LLM Agent v3 (full 434) | TBD | TBD | TBD |

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text: str):
    return model.encode(str(text), normalize_embeddings=True)


print("Embedding model ready")

In [ ]:
COMPONENT_ALIASES = {
    "md_max72xx": "arduino_led_matrix",
    "max7219": "arduino_led_matrix",
    "ws2812": "adafruit_neopixel",
    "neopixel": "adafruit_neopixel",
    "esp8266wifi": "wifi",
    "esp32wifi": "wifi",
}

In [ ]:
BOARD_KEYWORDS = [
    "arduino", "esp32", "esp8266", "atmega", "raspberry pi", "microcontroller"
]

def is_board(name):
    name = name.lower()
    return any(k in name for k in BOARD_KEYWORDS)

In [ ]:
# NOTE: DEFAULT_COMPONENTS and CONDITIONAL_DEFAULTS were used by an
# earlier abandoned version of build_project_query. The active query
# builder (build_weighted_queries in Cell 14) uses DOMAIN_MAP instead.
# These constants are kept here for reference but are NOT used in the pipeline.
DEFAULT_COMPONENTS = ["led", "resistor", "power supply"]  # unused
CONDITIONAL_DEFAULTS = {                                   # unused
    "button": ["button", "switch"],
    "sound":  ["buzzer", "speaker"],
    "display": ["lcd", "oled", "display"],
}


In [ ]:
def normalize_component_name(name):
    n = str(name); nl = n.lower()
    if "max30100" in nl or "max30102" in nl or "heart" in nl: return n + " heart rate pulse sensor"
    if "mq-3" in nl: return n + " alcohol gas sensor"
    if "mq" in nl and "sensor" in nl: return n + " gas sensor"
    if "mpu6050" in nl: return n + " accelerometer gyroscope sensor"
    if "ultrasonic" in nl: return n + " distance sensor"
    if "pir" in nl: return n + " motion sensor"
    if "neo-6m" in nl or "gps" in nl: return n + " gps module location sensor"
    if "hc-05" in nl or "hc-06" in nl: return n + " bluetooth module"
    if "nrf" in nl: return n + " rf wireless module"
    if "oled" in nl: return n + " display screen"
    if "lcd" in nl: return n + " display screen"
    if "servo" in nl: return n + " motor actuator"
    if "relay" in nl: return n + " switch module"
    return n


def build_component_text(row):
    parts = []
    name = str(row.get("Component name", ""))
    if name and name != "nan": parts.append(f"{name} {name}")
    category = str(row.get("Category", ""))
    if category and category != "nan": parts.append(category)
    subcat = str(row.get("Sub-category", ""))
    if subcat and subcat != "nan": parts.append(subcat)
    use_cases = str(row.get("Typical Use Cases", ""))
    if use_cases and use_cases != "nan": parts.append(use_cases)
    libs = str(row.get("Associated Libraries", ""))
    if libs and libs != "nan": parts.append(f"{libs} {libs}")
    return " | ".join(parts)


DOMAIN_MAP = {
    "turbidity":   ["turbidity sensor"],
    "drowsiness":  ["camera"],
    "ultrasonic":  ["ultrasonic sensor"],
    "rf":          ["rf module"],
    "gps":         ["gps module"],
    "heart":       ["pulse sensor"],
    "alcohol":     ["mq-3 sensor"],
    "gas":         ["gas sensor"],
    "motion":      ["pir sensor"],
    "soil moisture":   ["soil moisture sensor"],
    "temperature":     ["temperature sensor", "dht sensor"],
    "humidity":        ["dht sensor"],
    "bluetooth":       ["bluetooth module"],
    "smoke":           ["mq sensor"],
    "fire":            ["flame sensor"],
    "water level":     ["water level sensor"],
    "fingerprint":     ["fingerprint sensor"],
    "color":           ["color sensor"],
    "speed":           ["encoder", "ir sensor"],
    "line follow":     ["ir sensor"],
    "obstacle":        ["ultrasonic sensor", "ir sensor"],
    "servo":           ["servo motor"],
    "stepper":         ["stepper motor"],
    "oled":            ["oled display"],
    "lcd":             ["lcd display"],
    "led strip":       ["neopixel"],
    "relay":           ["relay module"],
    "sd card":         ["sd card module"],
    "rfid":            ["rfid reader"],
}

def expand_domain(description):
    text = description.lower()
    found = []
    for key, exps in DOMAIN_MAP.items():
        if key in text: found.extend(exps)
    return list(set(found))


def build_project_query(project):
    desc = str(project.get("description", ""))
    libs = project.get("ground_truth_libraries", [])
    parts = [desc] if desc else []
    hints = expand_domain(desc)
    if hints: parts.append(" ".join(hints))
    if isinstance(libs, list) and libs:
        parts.append(" ".join(libs) + " " + " ".join(libs * 3))
    return " | ".join(parts)


def build_weighted_queries(project):
    """Returns list of (query, weight) tuples for multi-intent retrieval."""
    desc = str(project.get("description", ""))
    libs = project.get("ground_truth_libraries", [])
    queries = [(build_project_query(project), 1.0)]
    if isinstance(libs, list):
        for lib in libs:
            if lib: queries.append((lib, 0.7))
    for hint in expand_domain(desc):
        queries.append((hint, 0.3))
    return queries


def prepare_component_embeddings(df_components):
    df_components = df_components.copy()
    df_components["Component name"] = df_components["Component name"].apply(normalize_component_name)
    df_components["combined_text"] = df_components.apply(build_component_text, axis=1)
    texts = df_components["combined_text"].tolist()
    embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    return df_components, embeddings


# Run once
df_components, comp_embeddings = prepare_component_embeddings(df_comp)
print("Component embeddings ready:", len(comp_embeddings))

In [ ]:
# ── CELL 8: BUILD BM25 INDEX ──────────────────────────────────────────────────
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9_ ]', ' ', text)  
    return text.split()

bm25_corpus = [tokenize(row["combined_text"]) for _, row in df_components.iterrows()]
bm25        = BM25Okapi(bm25_corpus)

print("BM25 index built over", len(bm25_corpus), "components")

In [ ]:
def get_best_components(project_query, comp_embeddings, df_components, top_k=5, candidate_pool=20):
    # 1. Embedding scores
    query_emb  = model.encode([project_query], normalize_embeddings=True)
    emb_scores = sklearn_cosine_similarity(query_emb, comp_embeddings)[0]

    # 2. BM25 scores
    bm25_scores = np.array(bm25.get_scores(tokenize(project_query)))

    # 3. Normalise both to [0, 1]
    def norm(arr):
        mn, mx = arr.min(), arr.max()
        return (arr - mn) / (mx - mn + 1e-9)

    hybrid = 0.6 * norm(emb_scores) + 0.4 * norm(bm25_scores)

    # 4. Candidate pool
    candidate_indices = np.argsort(hybrid)[::-1][:candidate_pool]

    # 5. Re-rank: bonus if component name appears verbatim in query
    query_lower = project_query.lower()
    reranked = []
    for idx in candidate_indices:
        comp_name = str(df_components.iloc[idx].get("Component name", "")).lower()
        bonus     = 0.15 if comp_name and comp_name in query_lower else 0.0
        reranked.append((idx, hybrid[idx] + bonus))

    reranked.sort(key=lambda x: x[1], reverse=True)

    results = []
    for idx, score in reranked[:top_k]:
        results.append({
            "Component_Id":   int(df_components.iloc[idx]["Component_Id"]),
            "Component_name": df_components.iloc[idx]["Component name"],
            "score":          float(score),
        })

    return results

In [ ]:
def get_confidence(score):
    if score > 0.65:
        return "high"
    elif score > 0.45:
        return "medium"
    else:
        return "low"

In [ ]:
COMMON_FP_IDS = {87, 88, 96, 92, 171} 

In [ ]:
# Build from raw df_comp (before normalize_component_name mutates names)
# so filter checks like is_board() operate on original dictionary names.
comp_name_lookup = {
    int(row["Component_Id"]): row["Component name"]
    for _, row in df_comp.iterrows()
}
print(f"comp_name_lookup built: {len(comp_name_lookup)} entries")


In [ ]:
def process_project(project, df_components, comp_embeddings, comp_name_lookup):
    # 1. Build weighted multi-queries
    weighted_queries = build_weighted_queries(project)

    # 2. Accumulate scores across all queries
    n_comp = len(df_components)
    all_scores = np.zeros(n_comp)

    for q, weight in weighted_queries:
        q_emb = model.encode([q], normalize_embeddings=True)
        e_sc  = sklearn_cosine_similarity(q_emb, comp_embeddings)[0]
        b_sc  = np.array(bm25.get_scores(tokenize(q)))

        def norm(a):
            mn, mx = a.min(), a.max()
            return (a - mn) / (mx - mn + 1e-9)

        hybrid = 0.6 * norm(e_sc) + 0.4 * norm(b_sc)
        top_idx = np.argsort(hybrid)[::-1][:20]   
        for idx in top_idx:
            all_scores[idx] += hybrid[idx] * weight

    # 3. Global ranking → top-25 candidates
    cand_indices = np.argsort(all_scores)[::-1][:25]
    matches = []
    for idx in cand_indices:
        matches.append({
            "Component_Id":   int(df_components.iloc[idx]["Component_Id"]),
            "Component_name": df_components.iloc[idx]["Component name"],
            "score":          float(all_scores[idx]),
        })

    # 4. Rerank
    query_str = " ".join(q for q, _ in weighted_queries)
    matches = rerank_with_keywords(matches, query_str)
    matches = rerank_with_penalty(matches)

    # 5. Filter boards + noise
    NOISE_WORDS = ["wire", "jumper", "resistor", "capacitor", "breadboard", "usb cable"]  # FIX 2: added usb cable
    filtered = [
        m for m in matches
        if not is_board(comp_name_lookup[m["Component_Id"]])
        and not any(nw in comp_name_lookup[m["Component_Id"]].lower() for nw in NOISE_WORDS)
    ]

    # FIX 3: score threshold — only keep components with meaningful scores
    MIN_SCORE = 0.3
    filtered_by_score = [m for m in filtered if m["score"] >= MIN_SCORE]

    # FIX 4: fallback if nothing passes threshold
    if not filtered_by_score:
        filtered_by_score = filtered[:3]

    # FIX 5: cap at 7, consistent everywhere
    top_filtered = filtered_by_score[:7]

    predicted_ids  = [m["Component_Id"] for m in top_filtered]  
    confidence_map = {
        str(m["Component_Id"]): get_confidence(m["score"])
        for m in top_filtered                                     
    }

    return {
        "project_id":                 project["project_id"],
        "board":                      project.get("board"),
        "board_fqbn":                 project.get("board_fqbn"),
        "complexity":                 project.get("complexity"),
        "ground_truth_component_ids": project.get("ground_truth_component_ids", []),
        "predicted_component_ids":    predicted_ids,
        "ground_truth_libraries":     project.get("ground_truth_libraries", []),
        "linking_confidence":         confidence_map,
        "llm_reasoning":              f"Top-{len(predicted_ids)} via weighted multi-query retrieval.",
        "manually_verified":          False,
        "retrieved_components":       top_filtered,             
    }

In [ ]:
def rerank_with_keywords(matches, query):
    query = query.lower()

    KEYWORDS = [
        "sensor", "rf", "wifi", "bluetooth", "motor",
        "display", "oled", "lcd", "camera", "gps",
        "accelerometer", "temperature", "ultrasonic",
        "relay", "led"
    ]

    boosted = []
    for m in matches:
        name = m.get("Component_name", "").lower()  # fixed: lowercase key matches get_best_components output
        score = m["score"]
        boost = 0
        for kw in KEYWORDS:
            if kw in name and kw in query:
                boost += 0.02
        m["score"] = score + boost
        boosted.append(m)

    return sorted(boosted, key=lambda x: x["score"], reverse=True)

In [ ]:
def rerank_with_penalty(matches):
    """Single authoritative version: suppresses known FP boards + penalises noise words."""
    PENALTY_WORDS = ["wire", "jumper", "resistor", "capacitor"]

    def score(m):
        name = m.get("Component_name", "").lower()
        penalty = -0.3 if m["Component_Id"] in COMMON_FP_IDS else 0.0
        for pw in PENALTY_WORDS:
            if pw in name:
                penalty -= 0.03
        return m["score"] + penalty

    return sorted(matches, key=score, reverse=True)

In [ ]:
INFRASTRUCTURE_IDS = {97, 98, 99, 100, 104, 115}  # breadboard, wires, resistor, power, usb, generic LED

results = []
error_count = 0

for proj in tqdm(projects, desc="Evaluating"):
    try:
        output = process_project(
            proj,
            df_components,
            comp_embeddings,
            comp_name_lookup
        )

        predicted_ids = output["predicted_component_ids"]

        ground_truth = set(output["ground_truth_component_ids"]) - INFRASTRUCTURE_IDS
        predicted    = set(predicted_ids) - INFRASTRUCTURE_IDS  

        # Metrics
        tp        = len(ground_truth & predicted)
        precision = tp / len(predicted)    if predicted    else 0
        recall    = tp / len(ground_truth) if ground_truth else 0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) else 0)

        results.append({
            "project_id":       proj["project_id"],
            "predicted_ids":    list(predicted),
            "ground_truth_ids": list(ground_truth),
            "precision":        precision,
            "recall":           recall,
            "f1":               f1,
        })

    except Exception as e:
        error_count += 1
        print(f"ERROR project {proj['project_id']}: {e}")
        continue

print(f"Collected : {len(results)} results")
print(f"Errors    : {error_count} skipped")
print(f"Coverage  : {len(results)}/{len(projects)} ({len(results)/len(projects):.1%})")

In [ ]:
output_path = "embedding_hybrid_results.json"

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved to", output_path)

In [ ]:
avg_p  = np.mean([r["precision"] for r in results])
avg_r  = np.mean([r["recall"]    for r in results])
avg_f1 = np.mean([r["f1"]        for r in results])
print(f"Avg Precision: {avg_p:.3f} | Avg Recall: {avg_r:.3f} | Avg F1: {avg_f1:.3f}")
print(f"Total projects evaluated: {len(results)}")

In [ ]:
print("\n=== FAILURE ANALYSIS ===\n")

low_f1 = [r for r in results if r["f1"] < 0.2]

print(f"Low-F1 cases: {len(low_f1)} / {len(results)}\n")

for i, r in enumerate(low_f1[:10]):  # show first 10
    pid = r["project_id"]
    
    proj = next(p for p in projects if p["project_id"] == pid)

    print(f"Project {pid}")
    print("Description:", proj.get("description", "")[:120])
    print("GT IDs:     ", r["ground_truth_ids"])
    print("Predicted:  ", r["predicted_ids"])
    print("Precision:", round(r["precision"], 2),
          "Recall:", round(r["recall"], 2),
          "F1:", round(r["f1"], 2))
    print("-" * 50)

## TRIAL: LLM-based extraction agent (Module 1)

In [ ]:
import os
import re
import json
import time as _time
import random
import difflib
from openai import OpenAI
from typing import Optional, Tuple

MODELS = {
    'llama3_3':   ('nvidia', 'meta/llama-3.3-70b-instruct',                  'NVIDIA_API_KEY_LLAMA33'),
    'mistral':    ('nvidia', 'mistralai/mistral-large-3-675b-instruct-2512', 'NVIDIA_API_KEY_MISTRAL'),
    'phi_medium': ('nvidia', 'microsoft/phi-3-medium-128k-instruct',         'NVIDIA_API_KEY_PHI3MEDIUM'),
    'gemma_27b':  ('nvidia', 'google/gemma-2-27b-it',                        'NVIDIA_API_KEY_GEMMA'),
    'phi_mini':   ('nvidia', 'microsoft/phi-3-mini-128k-instruct',           'NVIDIA_API_KEY_PHI3MINI'),
}

NVIDIA_BASE_URL  = 'https://integrate.api.nvidia.com/v1'
API_CALL_DELAY   = 1.0
API_TIMEOUT      = 30
MAX_RETRIES      = 5
MAX_PROMPT_CHARS = 32000

SYSTEM_PROMPT_MODULE1 = """You are a hardware component extraction specialist for Arduino and embedded systems projects.

Your task is to extract ALL hardware components from a natural language project description.
A component is any physical electronic part: sensors, actuators, modules, displays, motors, LEDs, relays, or communication devices.
Do NOT include boards (Arduino, ESP32, etc.), wires, resistors, capacitors, or breadboards.

Rules:
- Extract components explicitly named AND components implied by function
  (e.g. "measures temperature" → temperature sensor)
- Normalise synonyms to canonical names
  (e.g. "HC-SR04" → "Ultrasonic Sensor", "SG90" → "Servo Motor")
- If a library name is provided (e.g. "Servo.h", "DHT.h"), infer the associated component
- Output ONLY a JSON array. No explanation, no markdown, no preamble.

Confidence levels:
- "high"   → explicitly named or strong library signal
- "medium" → inferred from function description
- "low"    → inferred from domain context only

Output format:
[
  {
    "canonical_name": "Ultrasonic Sensor",
    "raw_mention": "HC-SR04",
    "confidence": "high"
  }
]

--- EXAMPLES ---

Example 1:
Description: "Smart dustbin that uses an ultrasonic sensor to detect nearby objects and opens the lid using a servo motor."
Libraries: Servo.h
Output:
[
  {"canonical_name": "Ultrasonic Sensor", "raw_mention": "ultrasonic sensor", "confidence": "high"},
  {"canonical_name": "Servo Motor",       "raw_mention": "servo motor",       "confidence": "high"}
]

Example 2:
Description: "Plant watering system that monitors soil moisture and controls a pump via Blynk, with LED matrix feedback."
Libraries: BlynkSimpleWifi, Arduino_LED_Matrix, EEPROM
Output:
[
  {"canonical_name": "Soil Moisture Sensor", "raw_mention": "soil moisture",   "confidence": "high"},
  {"canonical_name": "Relay Module",         "raw_mention": "controls a pump", "confidence": "medium"},
  {"canonical_name": "LED Matrix",           "raw_mention": "LED matrix",      "confidence": "high"}
]

Example 3:
Description: "Line follower robot using infrared sensors and DC motors controlled by an L293D motor driver."
Libraries: None
Output:
[
  {"canonical_name": "IR Sensor", "raw_mention": "infrared sensors", "confidence": "high"},
  {"canonical_name": "DC Motor",  "raw_mention": "DC motors",        "confidence": "high"}
]

--- END EXAMPLES ---"""

USER_PROMPT_MODULE1_TEMPLATE = """Extract all hardware components from the following Arduino project.

Project description:
{description}

Libraries used (if known):
{libraries}

Return ONLY the JSON array."""

class LLMClient:
    def __init__(self, model_key: str, temperature: float = 0):
        if model_key not in MODELS:
            raise ValueError(f'Unknown model_key: {model_key!r}. Valid: {list(MODELS.keys())}')

        self.model_key   = model_key
        self.provider, self.model_name, api_key_name = MODELS[model_key]
        self.temperature = temperature
        self.call_log    = []

        api_key = os.getenv(api_key_name)
        if not api_key:
            raise ValueError(f'Missing API key — set {api_key_name} in your .env file')

        self.client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=api_key)
        print(f'LLMClient ready: {model_key} ({self.model_name})')

    def generate(self, prompt: str, system_prompt: str) -> Tuple[str, float]:
        last_exc = None

        for attempt in range(MAX_RETRIES):
            try:
                _time.sleep(API_CALL_DELAY)
                t0 = _time.time()

                messages = (
                    [{'role': 'user', 'content': system_prompt + "\n\n" + prompt}]
                    if self.model_key == 'gemma_27b' else
                    [
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user',   'content': prompt},
                    ]
                )

                response = self.client.chat.completions.create(
                    model       = self.model_name,
                    messages    = messages,
                    temperature = self.temperature,
                    max_tokens  = 512,
                    timeout     = API_TIMEOUT,
                )

                latency = _time.time() - t0
                text    = response.choices[0].message.content.strip()

                self.call_log.append({
                    'latency':      round(latency, 3),
                    'attempt':      attempt + 1,
                    'success':      True,
                    'prompt_len':   len(prompt),
                    'response_len': len(text),
                })
                return text, latency

            except Exception as exc:
                last_exc = exc
                if attempt < MAX_RETRIES - 1:
                    wait = (15.0 * (2 ** attempt) + random.uniform(1, 3)
                            if '429' in str(exc) else 0.5 * (attempt + 1))
                    print(f'  Attempt {attempt+1}/{MAX_RETRIES} failed: {exc} — retrying in {wait:.1f}s')
                    _time.sleep(wait)

        self.call_log.append({'latency': 0.0, 'success': False, 'error': str(last_exc)[:100]})
        print(f'LLM call FAILED after {MAX_RETRIES} retries: {last_exc}')
        return '', 0.0


def call_llm(prompt: str, model, system_prompt: str = SYSTEM_PROMPT_MODULE1) -> Tuple[any, float]:
    if model == 'stub':
        return [], 0.0

    if not isinstance(model, LLMClient):
        raise TypeError(f'model must be "stub" or LLMClient instance, got {type(model).__name__}')

    if len(prompt) > MAX_PROMPT_CHARS:
        prompt = prompt[:MAX_PROMPT_CHARS - 200] + '\n\n[TRUNCATED]'

    raw, latency = model.generate(prompt, system_prompt=system_prompt)

    if not raw:
        return [], latency

    text = raw.strip()
    text = re.sub(r'^```(?:json)?\n?', '', text)
    text = re.sub(r'\n?```$',          '', text).strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed, latency
        if isinstance(parsed, dict) and 'components' in parsed:
            return parsed['components'], latency
    except Exception:
        pass

    match = re.search(r'\[.*?\]', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0)), latency
        except Exception:
            pass

    print(f'  WARNING: Could not parse LLM response:\n{text[:300]}')
    return [], latency


def module1_lookup(canonical_name: str, df_components, threshold: float = 0.40) -> Optional[int]:
    name_lower  = canonical_name.lower().strip()
    name_tokens = set(name_lower.split())
    best_score  = 0.0
    best_id     = None

    for _, row in df_components.iterrows():
        comp_name   = str(row["Component name"]).lower().strip()
        comp_tokens = set(comp_name.split())
        cid         = int(row["Component_Id"])

        if name_lower == comp_name:
            return cid
        if name_lower in comp_name:
            score = 0.95
        elif comp_name in name_lower:
            score = 0.90
        else:
            overlap = (len(name_tokens & comp_tokens) / len(name_tokens | comp_tokens)
                       if name_tokens and comp_tokens else 0.0)
            fuzzy   = difflib.SequenceMatcher(None, name_lower, comp_name).ratio()
            score   = max(overlap, fuzzy)

        if score > best_score:
            best_score = score
            best_id    = cid

    return best_id if best_score >= threshold else None


# ── Verify ────────────────────────────────────────────────────────────────────
print("── API Key Check ──")
for key, (_, _, env_var) in MODELS.items():
    status = "Yes" if os.getenv(env_var) else " MISSING"
    print(f"  {key:<12} {env_var:<35} {status}")


In [ ]:
active_client = LLMClient('llama3_3')

In [ ]:
INFRASTRUCTURE_NAMES = {
    "Breadboard", "Jumper Wires", "Power Supply",
    "Resistor", "USB Cable", "LED"   
}

def get_gt_names(project: dict) -> set:
    """Convert ground truth component IDs → canonical names, strip infrastructure."""
    gt_ids   = project.get("ground_truth_component_ids", [])
    gt_names = set()
    for cid in gt_ids:
        entry = comp_lookup.get(int(cid))
        if entry:
            name = str(entry.get("Component name", "")).strip()
            if name and name not in INFRASTRUCTURE_NAMES:
                gt_names.add(name.lower())
    return gt_names

for proj in projects[:3]:
    print(f"Project {proj['project_id']}: {get_gt_names(proj)}")

In [ ]:
COMPONENT_SYNONYMS = {
    "ir sensor":          ["line finder", "infrared sensor", "line follower sensor"],
    "line finder":        ["ir sensor", "infrared sensor"],
    "ultrasonic sensor":  ["distance sensor", "hc-sr04", "urm09"],
    "soil moisture":      ["moisture sensor", "grove moisture"],
    "relay":              ["relay module"],
    "dc motor":           ["motor with h-bridge", "gear motor"],
    "servo":              ["servo motor"],
    "bluetooth":          ["hc-05", "hc-06", "bluetooth module"],
    "oled":               ["oled display", "ssd1306"],
}

def names_match(pred_name: str, gt_name: str, threshold: float = 0.60) -> bool:
    p = pred_name.lower().strip()
    g = gt_name.lower().strip()

    if p == g:                 return True
    if p in g or g in p:      return True

    for key, synonyms in COMPONENT_SYNONYMS.items():
        if key in p or key in g:
            all_names = [key] + synonyms
            if any(s in p or s in g for s in all_names):
                return True

    p_tok = set(p.split())
    g_tok = set(g.split())
    if p_tok and g_tok:
        overlap = len(p_tok & g_tok) / len(p_tok | g_tok)
        if overlap >= threshold:
            return True

    if difflib.SequenceMatcher(None, p, g).ratio() >= threshold:
        return True

    return False


def match_predicted_to_gt(predicted_names: list, gt_names: set) -> dict:
    """
    For each predicted name, check if it matches any GT name.
    Returns tp, fp, fn counts.
    """
    matched_gt = set()
    tp = 0

    for pred in predicted_names:
        for gt in gt_names:
            if gt not in matched_gt and names_match(pred, gt):
                tp += 1
                matched_gt.add(gt)
                break

    fp = len(predicted_names) - tp
    fn = len(gt_names) - tp

    return {"tp": tp, "fp": fp, "fn": fn, "matched_gt": matched_gt}

In [ ]:
def process_project_agent(project, df_components, client):
    desc    = str(project.get("description", ""))
    libs    = project.get("ground_truth_libraries", [])
    lib_str = ", ".join(libs) if libs else "None"

    user_prompt = USER_PROMPT_MODULE1_TEMPLATE.format(
        description = desc,
        libraries   = lib_str,
    )

    extractions, latency = call_llm(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_MODULE1,
    )

    # Extract names directly — no ID mapping
    predicted_names = []
    for item in (extractions if isinstance(extractions, list) else []):
        if not isinstance(item, dict):
            continue
        name = item.get("canonical_name", "").strip()
        conf = item.get("confidence", "low")
        if name and conf in ("high", "medium"):  
            predicted_names.append(name)

    seen = set()
    predicted_names = [x for x in predicted_names if not (x.lower() in seen or seen.add(x.lower()))]

    return {
        "project_id":        project["project_id"],
        "predicted_names":   predicted_names,
        "extractions":       extractions if isinstance(extractions, list) else [],
        "latency":           latency,
    }

In [ ]:
checks = {
    "LLMClient":                    "LLMClient" in dir(),
    "call_llm":                     "call_llm" in dir(),
    "SYSTEM_PROMPT_MODULE1":        "SYSTEM_PROMPT_MODULE1" in dir(),
    "USER_PROMPT_MODULE1_TEMPLATE": "USER_PROMPT_MODULE1_TEMPLATE" in dir(),
    "module1_lookup":               "module1_lookup" in dir(),
    "process_project_agent":        "process_project_agent" in dir(),
    "get_gt_names":                 "get_gt_names" in dir(),
    "names_match":                  "names_match" in dir(),
    "match_predicted_to_gt":        "match_predicted_to_gt" in dir(),
    "projects":                     "projects" in dir(),
    "df_components":                "df_components" in dir(),
    "comp_lookup":                  "comp_lookup" in dir(),
    "active_client":                "active_client" in dir(),
}

for name, loaded in checks.items():
    print(f"{'Yes' if loaded else 'Missing'} {name}")

In [ ]:
agent_results = []
error_count   = 0

for proj in tqdm(projects[:3], desc=f"Agent [{active_client.model_key}]"):
    try:
        output   = process_project_agent(proj, df_components, active_client)
        gt_names = get_gt_names(proj)

        if not gt_names:
            continue

        scores = match_predicted_to_gt(output["predicted_names"], gt_names)

        tp, fp, fn = scores["tp"], scores["fp"], scores["fn"]
        precision  = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall     = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1         = (2 * precision * recall / (precision + recall)
                      if (precision + recall) > 0 else 0)

        agent_results.append({
            "project_id":      proj["project_id"],
            "predicted_names": output["predicted_names"],
            "gt_names":        list(gt_names),
            "matched":         list(scores["matched_gt"]),
            "precision":       precision,
            "recall":          recall,
            "f1":              f1,
            "latency":         output["latency"],
        })

    except Exception as e:
        error_count += 1
        print(f"ERROR project {proj['project_id']}: {e}")
        continue

# ── Results ───────────────────────────────────────────────────────────────────
avg_p   = np.mean([r["precision"] for r in agent_results])
avg_r   = np.mean([r["recall"]    for r in agent_results])
avg_f1  = np.mean([r["f1"]        for r in agent_results])
avg_lat = np.mean([r["latency"]   for r in agent_results])

print(f"Model      : {active_client.model_key}")
print(f"Projects   : {len(agent_results)} evaluated | {error_count} errors")
print(f"Precision  : {avg_p:.3f}")
print(f"Recall     : {avg_r:.3f}")
print(f"F1         : {avg_f1:.3f}")
print(f"Avg Latency: {avg_lat:.2f}s")

print("\n── Per Project ──")
for r in agent_results:
    print(f"\n  Project {r['project_id']}")
    print(f"  GT        : {r['gt_names']}")
    print(f"  Predicted : {r['predicted_names']}")
    print(f"  Matched   : {r['matched']}")
    print(f"  P:{r['precision']:.2f}  R:{r['recall']:.2f}  F1:{r['f1']:.2f}")

In [ ]:
SAVE_PATH = "agent_results_llama3_3.json"

agent_results = []
error_count   = 0

for proj in tqdm(projects, desc=f"Agent [{active_client.model_key}]"):
    try:
        output   = process_project_agent(proj, df_components, active_client)
        gt_names = get_gt_names(proj)

        if not gt_names:
            continue

        scores    = match_predicted_to_gt(output["predicted_names"], gt_names)
        tp, fp, fn = scores["tp"], scores["fp"], scores["fn"]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0)

        agent_results.append({
            "project_id":      proj["project_id"],
            "predicted_names": output["predicted_names"],
            "gt_names":        list(gt_names),
            "matched":         list(scores["matched_gt"]),
            "precision":       precision,
            "recall":          recall,
            "f1":              f1,
            "latency":         output["latency"],
        })

        # ── Save every 10 projects ──────────────────────────────────────────
        if len(agent_results) % 10 == 0:
            with open(SAVE_PATH, "w") as f:
                json.dump(agent_results, f, indent=2)
            tqdm.write(f"  💾 Saved {len(agent_results)} results so far...")

    except Exception as e:
        error_count += 1
        tqdm.write(f"  ERROR project {proj['project_id']}: {e}")
        continue

# ── Final save ────────────────────────────────────────────────────────────────
with open(SAVE_PATH, "w") as f:
    json.dump(agent_results, f, indent=2)

avg_p   = np.mean([r["precision"] for r in agent_results])
avg_r   = np.mean([r["recall"]    for r in agent_results])
avg_f1  = np.mean([r["f1"]        for r in agent_results])
avg_lat = np.mean([r["latency"]   for r in agent_results])

print(f"Model      : {active_client.model_key}")
print(f"Projects   : {len(agent_results)} evaluated | {error_count} errors")
print(f"Precision  : {avg_p:.3f}")
print(f"Recall     : {avg_r:.3f}")
print(f"F1         : {avg_f1:.3f}")
print(f"Avg Latency: {avg_lat:.2f}s")
print(f"Saved to   : {SAVE_PATH}")

In [ ]:
low_f1 = [r for r in agent_results if r["f1"] < 0.2]
print(f"Low-F1 cases : {len(low_f1)} / {len(agent_results)}")
print(f"Avg Precision: {np.mean([r['precision'] for r in agent_results]):.3f}")
print(f"Avg Recall   : {np.mean([r['recall']    for r in agent_results]):.3f}")
print(f"Avg F1       : {np.mean([r['f1']        for r in agent_results]):.3f}")

print("\n── Sample Failures ──")
for r in low_f1[:5]:
    print(f"\n  Project {r['project_id']}")
    print(f"  GT        : {r['gt_names']}")
    print(f"  Predicted : {r['predicted_names']}")
    print(f"  Matched   : {r['matched']}")
    print(f"  F1        : {r['f1']:.2f}")

# 📚 Task 2: Dictionary Retrieval Accuracy
> **Measures:** How accurately the dictionary maps ground-truth component IDs to their correct libraries.

**Method:** For each project, take ground-truth component IDs → look up in dictionary → 
retrieve associated libraries → normalize both sides → compare against project's actual libraries.

**Key insight:** This is a pure dictionary quality test — no LLM involved. 
It answers: "If we know the exact components, does our dictionary give us the right libraries?"

**Status:** 🔄 In progress

In [ ]:
for cid, comp in list(comp_lookup.items())[:]:
    lib = comp.get("Associated Libraries")
    if str(lib) != "nan":
        print(f"ID {cid}: {lib}")

In [ ]:
def lookup_libs_by_name(component_name: str, df_components) -> dict:
    """
    Given a component name (from LLM):
    1. Fuzzy match against dictionary component names
    2. Retrieve associated libraries
    3. Strip platform libs
    Returns matched component info + libraries.
    """
    name_lower  = component_name.lower().strip()
    name_tokens = set(name_lower.split())

    best_score = 0.0
    best_row   = None

    for _, row in df_components.iterrows():
        comp_name   = str(row["Component name"]).lower().strip()
        comp_tokens = set(comp_name.split())

        # Strategy 1 — exact
        if name_lower == comp_name:
            best_row = row
            break

        # Strategy 2 — substring
        if name_lower in comp_name or comp_name in name_lower:
            score = 0.95
        else:
            # Strategy 3 — token overlap
            overlap = (len(name_tokens & comp_tokens) / len(name_tokens | comp_tokens)
                       if name_tokens and comp_tokens else 0.0)
            # Strategy 4 — fuzzy
            fuzzy   = difflib.SequenceMatcher(None, name_lower, comp_name).ratio()
            score   = max(overlap, fuzzy)

        if score > best_score:
            best_score = score
            best_row   = row

    # ── No match found ────────────────────────────────────────────────────────
    if best_row is None or best_score < 0.40:
        return {
            "matched_name": None,
            "match_score":  0.0,
            "libs":         [],
            "status":       "no_match",
        }

    # ── Match found — retrieve libraries ──────────────────────────────────────
    matched_name = str(best_row["Component name"])
    raw_libs     = best_row.get("Associated Libraries", "")

    # Edge case — no library field
    if not raw_libs or str(raw_libs).strip().lower() in ("nan", "none", ""):
        return {
            "matched_name": matched_name,
            "match_score":  round(best_score, 3),
            "libs":         [],
            "status":       "no_library",
        }

    # Parse + strip platform libs
    libs = parse_libraries(raw_libs)
    libs = {l for l in libs if normalize_lib(l) not in PLATFORM_LIBS}

    # Edge case — all libs were platform libs
    if not libs:
        return {
            "matched_name": matched_name,
            "match_score":  round(best_score, 3),
            "libs":         [],
            "status":       "platform_only",
        }

    # Edge case — multiple libs → take first
    if len(libs) > 1:
        libs = {sorted(libs)[0]}

    return {
        "matched_name": matched_name,
        "match_score":  round(best_score, 3),
        "libs":         list(libs),
        "status":       "ok",
    }


# ── Quick test ────────────────────────────────────────────────────────────────
test_names = [
    "Ultrasonic Sensor",
    "Servo Motor",
    "Soil Moisture Sensor",
    "Relay Module",
    "DHT Sensor",
    "LED Matrix",
    "Water Pump",
    "IR Sensor",
    "GPS Module",
    "Bluetooth Module",
    "RandomComponentThatDoesntExist",   # edge case
]

print("── lookup_libs_by_name test ──\n")
for name in test_names:
    result = lookup_libs_by_name(name, df_components)
    print(f"  {name:<35} → matched: {str(result['matched_name']):<45} "
          f"libs: {result['libs']}  status: {result['status']}")

In [ ]:
task2_results = []
skipped       = 0

for proj in tqdm(projects, desc="Task 2 — Dictionary Retrieval"):
    pid     = proj["project_id"]
    gt_libs = set(proj.get("ground_truth_libraries", []))

    # Use predicted names from Task 1 agent results
    agent_result = next(
        (r for r in agent_results if r["project_id"] == pid), None
    )

    # Skip if no Task 1 result or no GT libraries
    if not agent_result or not gt_libs:
        skipped += 1
        continue

    predicted_names = agent_result["predicted_names"]

    # ── Step 1: Look up libs for each predicted component name ────────────────
    retrieved_libs = set()
    lookup_details = []

    for name in predicted_names:
        result = lookup_libs_by_name(name, df_components)
        lookup_details.append({
            "predicted_name": name,
            "matched_name":   result["matched_name"],
            "match_score":    result["match_score"],
            "libs":           result["libs"],
            "status":         result["status"],
        })
        retrieved_libs.update(result["libs"])

    # ── Step 2: Canonicalize both sides ───────────────────────────────────────
    gt_canon  = canonicalize_set(gt_libs)
    ret_canon = canonicalize_set(retrieved_libs)

    # ── Step 3: Strip platform libs ───────────────────────────────────────────
    gt_canon  = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    # Skip if nothing scoreable
    if not gt_canon:
        skipped += 1
        continue

    # ── Step 4: Score ─────────────────────────────────────────────────────────
    tp = match_with_equivalence(gt_canon, ret_canon)

    precision = tp / len(ret_canon) if ret_canon else 0
    recall    = tp / len(gt_canon)  if gt_canon  else 0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) else 0)

    task2_results.append({
        "project_id":     pid,
        "gt_libs":        list(gt_canon),
        "retrieved_libs": list(ret_canon),
        "tp":             tp,
        "precision":      round(precision, 4),
        "recall":         round(recall,    4),
        "f1":             round(f1,        4),
        "lookup_details": lookup_details,
    })

print(f"\nEvaluated : {len(task2_results)} projects")
print(f"Skipped   : {skipped} projects")

In [ ]:
import re

def normalize_lib(lib: str) -> str:
    """Strip #include syntax, angle brackets, quotes, .h extension → lowercase bare name."""
    if not lib:
        return ""
    lib = lib.lower().strip()
    lib = re.sub(r'#include\s*[<"]', '', lib)
    lib = re.sub(r'[>"]', '', lib)
    lib = lib.replace('.h', '')
    return lib.strip()


def split_libs(lib: str) -> list:
    """Split a library string on / | or , into individual names."""
    return [l.strip() for l in re.split(r'[/|,]', str(lib)) if l.strip()]


def parse_libraries(lib_str: str) -> set:
    """Parse a comma-separated library string into a clean set."""
    if not lib_str or str(lib_str) == "nan":
        return set()
    return {l.strip() for l in str(lib_str).split(",") if l.strip()}


test_libs = [
    '#include <Servo.h>',
    'DHT.h',
    'Wire',
    'Adafruit_NeoPixel.h / FastLED.h',
    'BlynkSimpleWifi',
    'nan',
    '',
]

print("── normalize_lib test ──")
for lib in test_libs:
    print(f"  {lib!r:<40} → {normalize_lib(lib)!r}")

print("\n── split_libs test ──")
print(f"  'Servo.h / Wire.h' → {split_libs('Servo.h / Wire.h')}")
print(f"  'DHT.h | SPI.h'    → {split_libs('DHT.h | SPI.h')}")

print("\n── parse_libraries test ──")
print(f"  'Servo.h, Wire.h, DHT.h' → {parse_libraries('Servo.h, Wire.h, DHT.h')}")

In [ ]:
# Maps non-canonical library names → canonical equivalents
# Used to handle different naming conventions across projects

ALIASES = {
    "fastled library":                  "fastled",
    "neopixel":                         "adafruit_neopixel",
    "adafruit_neopixel":                "adafruit_neopixel",

    "adafruit_gfx":                     "adafruit_gfx",
    "adafruit_sh1106":                  "sh1106",
    "ssd1306_minimal":                  "ssd1306",
    "tinyozoled":                       "ssd1306",
    "spfd5408_adafruit_gfx":            "adafruit_gfx",
    "liquidcrystal_i2c":                "liquidcrystal",
    "liquidcrystal":                    "liquidcrystal",

    "adafruit_sensor":                  "adafruit_unified_sensor",
    "dht_u":                            "dht",
    "dht":                              "dht",
    "adafruit_mpu6050":                 "mpu6050",
    "sparkfunmlx90614":                 "mlx90614",
    "adafruit_bmp280":                  "bmp280",
    "adafruit_bme280":                  "bme280",

    "blynksimpleesp8266":               "blynk",
    "blynksimpleshieldesp8266_softser": "blynk",
    "blynksimplewifi":                  "blynk",
    "blynksimpleesp32":                 "blynk",
    "wificlientsecure":                 "wifi_generic",
    "wifi":                             "wifi_generic",
    "esp8266wifi":                      "wifi_generic",
    "esp32wifi":                        "wifi_generic",
    "wifis3":                           "wifi_generic",

    "tinygps++":                        "tinygps",
    "tinygpsplus":                      "tinygps",

    "nrf24l01":                         "rf24",
    "rf24audio":                        "rf24",

    "sx127x.lora":                      "lora",

    "mfrc522":                          "rc522",

    "adafruit_mqtt_client":             "adafruit_mqtt",

    "firebaseesp8266":                  "firebase",
    "firebaseesp32":                    "firebase",

    "espasynctcp":                      "espasyncwebserver",
}


test_aliases = [
    "BlynkSimpleWifi",
    "TinyGPS++",
    "MFRC522",
    "neopixel",
    "DHT_U",
    "FirebaseESP32",
]

print("── Alias resolution test ──")
for lib in test_aliases:
    norm    = normalize_lib(lib)
    aliased = ALIASES.get(norm, norm)
    print(f"  {lib:<30} → norm: {norm:<30} → alias: {aliased}")

In [ ]:
# Filters out non-Arduino libraries
BAD_KEYWORDS = ["python", "sdk", "matlab", "ros", "dependency", "pip", "npm"]

def is_valid_lib(lib: str) -> bool:
    """Return False if lib string contains non-Arduino keywords."""
    return not any(b in lib for b in BAD_KEYWORDS)


def canonicalize_set(lib_set: set) -> set:
    """
    Full normalization pipeline for a set of library strings:
    1. Split on / | ,
    2. normalize_lib → lowercase, strip .h and #include
    3. Filter empty + invalid
    4. Apply ALIASES
    Returns a clean set of canonical library names.
    """
    final = set()

    for lib in lib_set:
        parts = split_libs(lib)

        for p in parts:
            norm = normalize_lib(p)

            if not norm:
                continue

            if not is_valid_lib(norm):
                continue

            norm = ALIASES.get(norm, norm)

            final.add(norm)

    return final


test_sets = [
    {"Servo.h", "Wire.h", "DHT.h"},
    {"BlynkSimpleWifi", "Arduino_LED_Matrix", "EEPROM"},
    {"TinyGPS++", "SoftwareSerial"},
    {"FirebaseESP32", "WiFiClientSecure"},
    {"python-serial", "ros_lib"},          
    {"Adafruit_NeoPixel.h / FastLED.h"},   
]

print("── canonicalize_set test ──")
for s in test_sets:
    result = canonicalize_set(s)
    print(f"  Input  : {s}")
    print(f"  Output : {result}")
    print()

In [ ]:
LIB_EQUIVALENTS = {
    "dht":              {"dht", "dht11", "dht22", "dht_u"},
    "tinygps":          {"tinygps", "tinygps++", "tinygpsplus", "gps"},
    "rf24":             {"rf24", "nrf24l01", "rf24audio"},
    "rc522":            {"rc522", "mfrc522"},
    "ssd1306":          {"ssd1306", "oled", "adafruit_ssd1306"},
    "liquidcrystal":    {"liquidcrystal", "liquidcrystal_i2c"},
    "wifi_generic":     {"wifi_generic", "wifi", "esp8266wifi", "esp32wifi", "wifis3"},
    "adafruit_neopixel":{"adafruit_neopixel", "neopixel", "fastled"},
    "blynk":            {"blynk", "blynksimplewifi", "blynksimpleesp8266", "blynksimpleesp32"},
    "servo":            {"servo"},
    "wire":             {"wire", "i2c"},
    "softwareserial":   {"softwareserial", "serial"},
    "adafruit_gfx":     {"adafruit_gfx", "gfx"},
    "firebase":         {"firebase", "firebaseesp8266", "firebaseesp32"},
    "tinygps":          {"tinygps", "tinygps++"},
    "mpu6050":          {"mpu6050", "adafruit_mpu6050"},
    "bmp280":           {"bmp280", "adafruit_bmp280"},
    "bme280":           {"bme280", "adafruit_bme280"},
    "lora":             {"lora", "sx127x.lora"},
    "irremote":         {"irremote", "irrecv", "ir"},
}


def match_with_equivalence(gt_set: set, pred_set: set) -> int:
    """
    Count matches between gt_set and pred_set using equivalence groups.
    Each GT lib can only be matched once.
    Each pred lib can only be used once.
    """
    matched  = 0
    used_gt  = set()
    used_pred= set()

    for g in gt_set:
        for p in pred_set:
            if p in used_pred or g in used_gt:
                continue

            if g == p:
                matched += 1
                used_gt.add(g)
                used_pred.add(p)
                break

            matched_via_equiv = False
            for group in LIB_EQUIVALENTS.values():
                if g in group and p in group:
                    matched += 1
                    used_gt.add(g)
                    used_pred.add(p)
                    matched_via_equiv = True
                    break

            if matched_via_equiv:
                break

    return matched


print("── match_with_equivalence test ──\n")

tests = [
    # (gt_set,                    pred_set,                  expected)
    ({"dht"},                     {"dht22"},                  1),  
    ({"servo", "wire"},           {"servo", "wire"},          2),   
    ({"liquidcrystal_i2c"},       {"liquidcrystal"},          1),   
    ({"blynk"},                   {"blynksimplewifi"},         1),   
    ({"servo", "dht"},            {"servo", "dht11", "wire"}, 2),   
    ({"firebase"},                {"ssd1306"},                0),   
]

for gt, pred, expected in tests:
    gt_canon   = canonicalize_set(gt)
    pred_canon = canonicalize_set(pred)
    result     = match_with_equivalence(gt_canon, pred_canon)
    status     = "Done" if result == expected else "Missing"
    print(f"  {status} GT: {gt_canon} | Pred: {pred_canon}")
    print(f"     matched={result} expected={expected}\n")

In [ ]:
PLATFORM_LIBS = {
    "spi", "wire", "eeprom", "softwareserial", "wifi_generic",
    "wifis3", "esp8266wifi", "esp32wifi", "ethernet", "sd",
    "arduinojson", "httpclient", "time", "rtclib",
    "freertos", "arduino", "wifi", "wificlientsecure",
    "blynk", "firebase", "mqtt", "pubsubclient",
}


def dictionary_rag_lookup(component_ids: list) -> dict:
    """
    Given a list of ground-truth component IDs:
    1. Look up each in comp_lookup
    2. Retrieve associated libraries
    3. Strip platform libraries
    4. Return retrieved libraries + component details
    """
    retrieved = {
        "libraries":   set(),
        "components":  [],
        "missing_ids": [],
    }

    for cid in component_ids:
        comp = comp_lookup.get(int(cid))

        if not comp:
            retrieved["missing_ids"].append(cid)
            continue

        name = comp.get("Component name", "Unknown")
        libs = parse_libraries(comp.get("Associated Libraries", ""))

        libs = {l for l in libs if normalize_lib(l) not in PLATFORM_LIBS}

        retrieved["components"].append({
            "id":   cid,
            "name": name,
            "libs": list(libs),
        })

        retrieved["libraries"].update(libs)

    return retrieved


print("── dictionary_rag_lookup test (fixed) ──\n")
for proj in projects[:3]:
    pid    = proj["project_id"]
    gt_ids = proj.get("ground_truth_component_ids", [])
    result = dictionary_rag_lookup(gt_ids)

    print(f"Project {pid}")
    print(f"  GT IDs      : {gt_ids}")
    print(f"  Missing IDs : {result['missing_ids']}")
    print(f"  Components  :")
    for c in result["components"]:
        print(f"    [{c['id']:>3}] {c['name']:<45} libs: {c['libs']}")
    print(f"  Raw libs retrieved: {result['libraries']}")
    print()

In [ ]:
task2_results = []

for proj in tqdm(projects, desc="Task 2 — Dictionary Retrieval"):
    pid     = proj["project_id"]
    gt_ids  = proj.get("ground_truth_component_ids", [])
    gt_libs = set(proj.get("ground_truth_libraries", []))

    if not gt_libs:
        continue

    # Step 1: Retrieve from dictionary 
    retrieved      = dictionary_rag_lookup(gt_ids)
    retrieved_libs = retrieved["libraries"]

    # Step 2: Canonicalize both sides 
    gt_canon  = canonicalize_set(gt_libs)
    ret_canon = canonicalize_set(retrieved_libs)

    #  Step 3: Strip platform libs from both sides 
    gt_canon  = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    if not gt_canon:
        continue

    # Step 4: Score 
    tp = match_with_equivalence(gt_canon, ret_canon)

    precision = tp / len(ret_canon) if ret_canon else 0
    recall    = tp / len(gt_canon)  if gt_canon  else 0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) else 0)

    task2_results.append({
        "project_id":  pid,
        "gt_libs":     list(gt_canon),
        "retrieved":   list(ret_canon),
        "tp":          tp,
        "precision":   round(precision, 4),
        "recall":      round(recall,    4),
        "f1":          round(f1,        4),
        "missing_ids": retrieved["missing_ids"],
        "components":  retrieved["components"],
    })

print(f"\nEvaluated : {len(task2_results)} projects")
print(f"Skipped   : {len(projects) - len(task2_results)} (no scoreable GT libraries)")

In [ ]:
avg_p  = np.mean([r["precision"] for r in task2_results])
avg_r  = np.mean([r["recall"]    for r in task2_results])
avg_f1 = np.mean([r["f1"]        for r in task2_results])

print("=" * 50)
print(" Task 2 — Dictionary Retrieval Accuracy")
print("=" * 50)
print(f"  Projects evaluated : {len(task2_results)}")
print(f"  Avg Precision      : {avg_p:.3f}")
print(f"  Avg Recall         : {avg_r:.3f}")
print(f"  Avg F1             : {avg_f1:.3f}")
print("=" * 50)

buckets = {"1.00": 0, "0.75-0.99": 0, "0.50-0.74": 0, "0.25-0.49": 0, "0.00-0.24": 0}
for r in task2_results:
    p = r["precision"]
    if p == 1.0:                  buckets["1.00"]      += 1
    elif p >= 0.75:               buckets["0.75-0.99"] += 1
    elif p >= 0.50:               buckets["0.50-0.74"] += 1
    elif p >= 0.25:               buckets["0.25-0.49"] += 1
    else:                         buckets["0.00-0.24"] += 1

print("\n── Precision Distribution ──")
for bucket, count in buckets.items():
    bar = "█" * (count // 5)
    print(f"  {bucket} : {count:>4} projects  {bar}")

print("\n── Sample Output (first 5) ──")
for r in task2_results[:5]:
    print(f"\n  Project {r['project_id']}")
    print(f"  GT libs    : {r['gt_libs']}")
    print(f"  Retrieved  : {r['retrieved']}")
    print(f"  TP         : {r['tp']}")
    print(f"  P: {r['precision']:.2f}  R: {r['recall']:.2f}  F1: {r['f1']:.2f}")
    if r["missing_ids"]:
        print(f"   Missing IDs in dict: {r['missing_ids']}")

In [ ]:
low_f1    = [r for r in task2_results if r["f1"] < 0.2]
zero_f1   = [r for r in task2_results if r["f1"] == 0.0]
perfect   = [r for r in task2_results if r["f1"] == 1.0]

print("=" * 55)
print(" Task 2 — Failure Analysis")
print("=" * 55)
print(f"  Perfect F1  (1.00)       : {len(perfect)}")
print(f"  Low F1      (< 0.20)     : {len(low_f1)}")
print(f"  Zero F1     (0.00)       : {len(zero_f1)}")
print("=" * 55)

from collections import Counter

missed_libs  = Counter()
extra_libs   = Counter()

for r in task2_results:
    gt_set  = set(r["gt_libs"])
    ret_set = set(r["retrieved"])
    for lib in gt_set  - ret_set: missed_libs[lib]  += 1
    for lib in ret_set - gt_set:  extra_libs[lib]   += 1

print("\n── Top 10 Most Missed Libraries (in GT, not retrieved) ──")
for lib, count in missed_libs.most_common(10):
    print(f"  {lib:<40} missed {count}x")

print("\n── Top 10 Most Over-Retrieved Libraries (retrieved, not in GT) ──")
for lib, count in extra_libs.most_common(10):
    print(f"  {lib:<40} extra  {count}x")

all_missing_ids = Counter()
for r in task2_results:
    for mid in r["missing_ids"]:
        all_missing_ids[mid] += 1

if all_missing_ids:
    print(f"\n── Component IDs missing from dictionary ({len(all_missing_ids)} unique) ──")
    for cid, count in all_missing_ids.most_common(10):
        print(f"  ID {cid:<6} missing in {count} projects")
else:
    print("\n No missing component IDs — dictionary covers all GT components")

if zero_f1:
    print(f"\n── Sample Zero-F1 Projects (first 5) ──")
    for r in zero_f1[:5]:
        print(f"\n  Project {r['project_id']}")
        print(f"  GT libs   : {r['gt_libs']}")
        print(f"  Retrieved : {r['retrieved']}")
        print(f"  Components: {[(c['name'], c['libs']) for c in r['components']]}")

In [ ]:
import json
from pathlib import Path

SAVE_PATH = Path("task2_dictionary_retrieval_results.json")

summary = {
    "task":              "Task 2 — Dictionary Retrieval Accuracy",
    "total_evaluated":   len(task2_results),
    "total_skipped":     len(projects) - len(task2_results),
    "avg_precision":     round(np.mean([r["precision"] for r in task2_results]), 4),
    "avg_recall":        round(np.mean([r["recall"]    for r in task2_results]), 4),
    "avg_f1":            round(np.mean([r["f1"]        for r in task2_results]), 4),
    "perfect_f1_count":  len([r for r in task2_results if r["f1"] == 1.0]),
    "zero_f1_count":     len([r for r in task2_results if r["f1"] == 0.0]),
    "low_f1_count":      len([r for r in task2_results if r["f1"] < 0.2]),
    "results":           task2_results,
}

with open(SAVE_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f" Saved → {SAVE_PATH}")
print(f"   Total projects : {summary['total_evaluated']}")
print(f"   Avg Precision  : {summary['avg_precision']}")
print(f"   Avg Recall     : {summary['avg_recall']}")
print(f"   Avg F1         : {summary['avg_f1']}")

print("""
── Update Task 2 markdown status to ──

**Status:**  Complete

| Metric    | Score |
|-----------|-------|
| Precision | {p}   |
| Recall    | {r}   |
| F1        | {f}   |
| Projects  | {n}   |
""".format(
    p = summary["avg_precision"],
    r = summary["avg_recall"],
    f = summary["avg_f1"],
    n = summary["total_evaluated"],
))

# 🧠 Task 3: Intent Understanding (Completeness & Accuracy)
> **Measures:** Whether the LLM correctly interprets the *project intent* — identifying all required hardware from natural language, including implied and inferred components.

**Method:** Module 1 (LLM extraction agent) outputs `canonical_name` + `confidence` JSON. Results are fuzzy-matched to the component dictionary via `module1_lookup`. Scored against ground truth using exact + relaxed matching.

**Signals used:** Project description, library names (as first-class input), few-shot examples.

**Status:** 🔄 Prompt designed, `run_module1` implemented — evaluation pending API run.

In [ ]:
# ── CELL 4: COMPONENT DICTIONARY ─────────────────────────────────────────────
class HardwareDictionary:
    """Authoritative source for hardware components, pins, and libraries."""
    def __init__(self, excel_path: str):
        self.df = pd.read_excel(excel_path)
        self.df.columns = self.df.columns.str.strip()
        self.comp_dict = {}
        self.alias_to_id = {}
        self._build()

    def _build(self):
        for _, row in self.df.iterrows():
            cid = int(row['Component_Id'])
            name = str(row['Component name']).strip()
            
            # Pins & Libraries
            pin_type = str(row.get('Pin Type', 'Digital')).strip()
            libs = str(row.get('Associated Libraries', '')).strip()
            funcs = str(row.get('Function Names', '')).strip()
            
            # Aliases
            raw_aliases = str(row.get('Common versions', '')) if pd.notna(row.get('Common versions')) else ''
            aliases = [a.strip() for a in raw_aliases.split(',') if a.strip() and a.strip() != 'nan']

            self.comp_dict[cid] = {
                'id': cid,
                'name': name,
                'aliases': aliases,
                'pin_type': pin_type,
                'libraries': [l.strip() for l in libs.split(',') if l.strip()],
                'functions': [f.strip() for f in funcs.split(',') if f.strip()]
            }
            
            # Map name and all aliases to ID for lookup
            self.alias_to_id[name.lower()] = cid
            for a in aliases:
                self.alias_to_id[a.lower()] = cid

    def lookup(self, identifier) -> dict:
        """Lookup by ID (int) or Name/Alias (str)."""
        if isinstance(identifier, int):
            return self.comp_dict.get(identifier)
        if isinstance(identifier, str):
            cid = self.alias_to_id.get(identifier.lower())
            return self.comp_dict.get(cid) if cid else None
        return None

    def get_prompt_context(self, ids=None) -> str:
        """Returns formatted context for the LLM."""
        targets = ids if ids else self.comp_dict.keys()
        lines = []
        for cid in sorted(targets):
            c = self.comp_dict[cid]
            alias_str = f" (aliases: {', '.join(c['aliases'])})" if c['aliases'] else ""
            lib_str = f" | Libs: {', '.join(c['libraries'])}" if c['libraries'] else ""
            pin_str = f" | Pins: {c['pin_type']}"
            lines.append(f"  [{cid}] {c['name']}{alias_str}{pin_str}{lib_str}")
        return "CANONICAL COMPONENT LIST:\n" + "\n".join(lines)

# Initialize global dictionary
hw_dict = HardwareDictionary(COMP_PATH)
component_dict = hw_dict.comp_dict
alias_to_id = hw_dict.alias_to_id
component_context = hw_dict.get_prompt_context()

print(f'Total components loaded: {len(component_dict)}')
print(component_context[:600], '...')


In [ ]:
# ── STEP 1.1: DICTIONARY AUDIT ──────────────────────────────────────────────\n
def run_dictionary_audit(h_dict: HardwareDictionary):\n
    print("=== DICTIONARY AUDIT ===")\n
    errors = []\n
    for cid, c in h_dict.comp_dict.items():\n
        # Critical checks: I2C/SPI must have libraries\n
        if c["pin_type"].upper() in ["I2C", "SPI"] and not c["libraries"]:\n
            errors.append(f"[MISSING LIB] ID {cid} ({c['name']}) is {c['pin_type']} but has no Associated Libraries")\n
        \n
        # Warning: Digital/Analog with no libraries (usually fine for raw pins)\n
        if not c["libraries"] and c["pin_type"].upper() not in ["DIGITAL", "ANALOG", "PWM"]:\n
            print(f"  [Note] ID {cid} ({c['name']}) has no libraries listed.")\n
            \n
    if not errors:\n
        print("✅ No critical errors found in dictionary system.")\n
    else:\n
        print(f"❌ Found {len(errors)} critical issues:")\n
        for e in errors: print(f"  {e}")\n
\n
run_dictionary_audit(hw_dict)

# ⚙️ Task 4: Hardware Feasibility Mapping
> **Measures:** Whether the predicted component set is *physically realizable* — correct pin types, compatible libraries, and no hardware conflicts for the target board.

**Method:** For each predicted component, cross-reference against the dictionary's `Pin Type`, `Associated Libraries`, and `Board` compatibility fields. Flag conflicts (e.g. two I²C devices on same address, PWM pin overuse) and missing library declarations.

**GT tiers:** Infrastructure components (power, wires) are scored separately from functional components.

**Status:** ❌ Not started.

## 📄 Benchmark Definition

We categorize components into two groups:

**(1) Infrastructure components** (e.g., power supply, jumper wires, resistors, generic LEDs), which are ubiquitous and provide limited semantic signal.

**(2) Functional components** (e.g., sensors, actuators, displays, communication modules), which are essential to system behavior and require genuine hardware understanding to identify.

To prevent inflated performance from trivial predictions, infrastructure components are **excluded from evaluation metrics**. All reported precision, recall, and F1 scores are computed over **functional components only**.

| Category | Examples | Scored? |
|----------|----------|--------|
| **Functional — Sensors** | Ultrasonic, DHT11, PIR, Accelerometer | ✅ Yes |
| **Functional — Actuators** | Servo, DC Motor, Stepper, Solenoid | ✅ Yes |
| **Functional — Displays** | OLED, LCD, TFT, LED Matrix | ✅ Yes |
| **Functional — Communication** | Bluetooth, WiFi, LoRa, GPS | ✅ Yes |
| **Functional — Boards** | Arduino Uno, ESP32, Nano | ✅ Yes |
| **Infrastructure** | Breadboard, Jumper Wires, Resistor, Power Supply, USB Cable, Single LED | ❌ Excluded |

# 📝 Task 5: Code Planning Quality
> **Measures:** Whether the LLM produces a *correct structural plan* before generating code — correct `#include` statements, pin assignments, setup/loop decomposition.

**Method:** Parse generated `.ino` for: (1) presence of all required library headers, (2) `void setup()` / `void loop()` structure, (3) pin assignment coverage for all predicted components. Scored against ground-truth component libraries.

**Key cell:** `extract_code_signals` (Cell 52) — used to verify structured signals in generated code.

**Status:** ❌ Not started.

In [ ]:
# ── CELL GT-VALIDATE: Enhanced GT Extraction + Scoring Tiers ─────────────────
import re
from collections import Counter

# ── 1. Non-hardware blacklist ─────────────────────────────────────────────────
BLACKLIST = {
    'logic', 'algorithm', 'communication', 'management', 'processing',
    'control system', 'function', 'protocol', 'interface', 'handling',
    'system', 'feature', 'operation', 'service', 'support', 'control',
    'notification', 'storage', 'state', 'data', 'code', 'payload',
    'request', 'response', 'connectivity', 'actuation', 'averaging',
    'navigation', 'movement', 'directional', 'conditional', 'serial',
    'gpio', 'pin', 'pins', 'gpio pins', 'i2c', 'spi', 'uart',
}

# ── 2. Component tiers ────────────────────────────────────────────────────────
# Infrastructure = always present, low signal value, excluded from scoring
INFRASTRUCTURE_IDS: Set[int] = {
    97,   # Breadboard
    98,   # Jumper Wires
    99,   # Power Supply / Battery
    100,  # Resistor
    104,  # USB Cable
    115,  # Single Color LED (too generic — matches nearly everything)
}

# ── 3. Manual alias → component_id map ───────────────────────────────────────
MANUAL_ALIASES: Dict[str, int] = {
    # Sensors
    'ultrasonic sensor':1, 'ultrasonic':1, 'hc-sr04':1, 'distance sensor':1,
    'soil moisture sensor':9, 'soil moisture':9, 'moisture sensor':18,
    'pir sensor':17, 'pir motion':17, 'motion sensor':17,
    'ir sensor':21, 'infrared sensor':21, 'ir receiver':29, 'ir remote':29,
    'tsop ir receiver':29, 'ir led':76, 'ir blaster':76,
    'light sensor':19, 'ldr':19, 'photoresistor':19,
    'hall effect sensor':22, 'hall sensor':22,
    'temperature sensor':12, 'thermistor':42, 'lm35':42,
    'dht11':14, 'dht22':71, 'dht sensor':14,
    'temperature and humidity sensor':14, 'temperature & humidity sensor':14,
    'humidity sensor':14, 'bmp280':25, 'barometer':25, 'barometric':69, 'bme280':15,
    'sound sensor':16, 'microphone':73, 'max9814':73,
    'flame sensor':52, 'gas sensor':24, 'mq':24,
    'touch sensor':70, 'capacitive touch':46, 'color sensor':36,
    'accelerometer':28, 'gyroscope':28, 'mpu6050':28, 'imu':28,
    'current sensor':151, 'acs712':151, 'ina219':151,
    'voltage sensor':152, 'load cell':129, 'hx711':129,
    'flex sensor':130, 'force sensor':131, 'fsr':131,
    'heartbeat sensor':74, 'pulse sensor':74, 'heart rate':37,
    'fingerprint':149, 'rfid':32, 'nfc':32, 'dust sensor':4, 'air quality sensor':26,
    # Displays (NOT generic LED)
    'oled':105, 'oled display':105, 'ssd1306':105,
    'tft display':106, 'tft lcd':106, 'tft screen':106, 'tft touchscreen':106, 'ili9341':106,
    'lcd display':68, 'lcd':68, '16x2 lcd':68, '16\u00d72 lcd':68,
    '7-segment display':107, '7-segment':107, 'seven segment':107,
    'led matrix':108, '8x8 led matrix':108, '8\u00d78 led matrix':108,
    'nokia 5110':109, 'e-paper':156, 'e-ink':156,
    # LEDs (specific types only)
    'neopixel':113, 'ws2812':113, 'ws2812b':113, 'led strip':113, 'addressable led':113,
    'rgb led':63, 'rgb led module':63,
    'laser':114, 'laser diode':114, 'uv led':116,
    # NOTE: 'led' alone is NOT mapped — too generic, causes 152+ false positives
    # Actuators
    'servo motor':117, 'servo':117,
    'stepper motor':118, 'stepper':118,
    'dc motor':119, 'dc motors':119, 'motor driver':119,
    'l298n':119, 'l298':119, 'l293':119, 'h-bridge':119,
    'brushless motor':120, 'bldc':120,
    'gear motor':121, 'solenoid':122, 'solenoid valve':126,
    'water pump':165, 'submersible pump':165, 'peristaltic pump':166,
    'air pump':167, 'compressor':167, 'cooling fan':168, 'fan':168,
    'electromagnet':164, 'heating element':125, 'peltier':124,
    # Sound
    'buzzer':110, 'piezo buzzer':72, 'active buzzer':72, 'passive buzzer':110,
    'speaker':65, 'mp3 module':111, 'mp3 player':111, 'mp3 sound module':111,
    'jq6500':111, 'dfplayer':111, 'amplifier':112, 'pam8610':112,
    # Communication
    'bluetooth module':136, 'bluetooth':136, 'hc-05':136, 'hc-06':136,
    'hm-10':137, 'ble module':137, 'bluetooth low energy':137,
    'nrf24l01':138, 'rf transceiver':138,
    '433 mhz':172, '433mhz':172, 'rf transmitter':172, 'rf receiver':173,
    'lora':140, 'xbee':141, 'zigbee':141, 'esp-now':142,
    'gsm module':143, 'sim800':143, 'sim800l':143,
    'lte module':144, 'sim7000':144,
    'gps module':145, 'gps':145,
    'wifi module':93, 'wifi':93, 'wi-fi':93,
    'can bus':150, 'mcp2515':150, 'rs485':169, 'max485':169, 'hc-12':139,
    # Boards
    'arduino uno':87, 'arduino nano':88, 'arduino mega':89, 'arduino leonardo':90,
    'arduino due':91, 'esp32':92, 'esp32-s3':92, 'esp8266':93,
    'arduino micro':94, 'arduino pro mini':95, 'pro mini':95,
    'arduino nano 33':96, 'attiny85':170, 'digispark':170,
    'atmega328p':171, 'atmega328':171,
    # Input devices
    'push button':56, 'push buttons':56, 'button':56,
    'keypad':146, 'matrix keypad':146, '4x4 keypad':146, 'analog keypad':146,
    'joystick':51, 'rotary encoder':82, 'potentiometer':50,
    'limit switch':153, 'toggle switch':101, 'slide switch':102, 'rocker switch':103,
    'camera':147, 'webcam':147, 'barcode scanner':148,
    # Passive / power / misc
    'relay':64, 'relay module':64, 'solid state relay':123, 'ssr':123,
    'power supply':99, 'battery':99, 'li-po battery':99,
    'resistor':100, 'resistors':100, 'breadboard':97,
    'jumper wire':98, 'jumper wires':98, 'usb cable':104,
    'sd card module':157, 'sd card':157, 'microsd':157,
    'eeprom':158, 'at24c256':158,
    'rtc module':159, 'real-time clock':159, 'ds3231':159,
    'buck converter':160, 'lm2596':160, 'boost converter':161, 'mt3608':161,
    'voltage regulator':162, 'ams1117':162, '7805':162, 'regulator':162,
    'battery charging':163, 'tp4056':163, 'charger':163,
    'shift register':108, '74hc595':108, 'vibration motor':66,
}

# ── 4. Build combined lookup (canonical + manual) ─────────────────────────────
all_alias_to_id: Dict[str, int] = {}
for cid, info in component_dict.items():
    all_alias_to_id[info['name'].lower().strip()] = cid
    for alias in info.get('aliases', []):
        if alias:
            all_alias_to_id[alias.lower().strip()] = cid
all_alias_to_id.update(MANUAL_ALIASES)

# Remove bare 'led' from aliases — too generic (matches in 152 projects)
all_alias_to_id.pop('led', None)

sorted_aliases = sorted(all_alias_to_id.keys(), key=len, reverse=True)
print(f'Total lookup entries: {len(sorted_aliases)}')

# ── 5. Text normaliser ────────────────────────────────────────────────────────
def normalize_text(text: str) -> str:
    text = text.lower()
    text = text.replace('\u00d7', 'x').replace('\u2013', '-').replace('\u2014', '-')
    return re.sub(r'\\s+', ' ', text).strip()

# ── 6. Enhanced extraction ────────────────────────────────────────────────────
def extract_gt_from_text(text: str) -> List[int]:
    if not text or not text.strip():
        return []
    text_norm = normalize_text(text)
    found_ids: Set[int] = set()
    for alias in sorted_aliases:
        if alias in BLACKLIST:
            continue
        if len(alias) < 3:
            continue
        if alias in text_norm:
            found_ids.add(all_alias_to_id[alias])
    return sorted(found_ids)

# ── 7. Apply ──────────────────────────────────────────────────────────────────
# `projects` is the unified list built in the setup cells.
# `ground_truth_component_ids` is already parsed from the JSON GT file.
# We re-derive `ground_truth_all` (with infrastructure) and `ground_truth`
# (infrastructure excluded) for scoring tier compatibility.
for proj in projects:
    # ground_truth_component_ids is the canonical GT from the JSON file
    all_ids = proj.get('ground_truth_component_ids', [])
    proj['ground_truth_all'] = all_ids
    proj['ground_truth']     = [c for c in all_ids if c not in INFRASTRUCTURE_IDS]

dataset = projects  # alias for compatibility with the rest of this cell

# ── 8. Stats ──────────────────────────────────────────────────────────────────
lengths_all  = [len(p['ground_truth_all']) for p in dataset]
lengths_core = [len(p['ground_truth'])     for p in dataset]
all_gt_ids   = [c for p in dataset for c in p['ground_truth']]
id_counts    = Counter(all_gt_ids)

n_all  = sum(1 for l in lengths_all  if l > 0)
n_core = sum(1 for l in lengths_core if l > 0)

print(f'\n{"=" * 55}')
print(f' GROUND TRUTH EXTRACTION REPORT')
print(f'{"=" * 55}')
print(f'  Total projects            : {len(dataset)}')
print(f'  --- All components (incl. infrastructure) ---')
print(f'  Non-empty GT (all)        : {n_all} ({n_all/len(dataset):.1%})')
print(f'  Avg components (all)      : {sum(lengths_all)/len(lengths_all):.2f}')
print(f'  --- Scoring components (excl. infrastructure) ---')
print(f'  Non-empty GT (scored)     : {n_core} ({n_core/len(dataset):.1%})')
print(f'  Avg components (scored)   : {sum(lengths_core)/len(lengths_core):.2f}')
print(f'  Max components (scored)   : {max(lengths_core)}')
print(f'  Unique scored comp IDs    : {len(set(all_gt_ids))}')
print(f'\n  Infrastructure (excluded): {[component_dict[c]["name"] for c in sorted(INFRASTRUCTURE_IDS)]}')

print(f'\n  Top 10 scored GT components:')
for cid, cnt in id_counts.most_common(10):
    print(f'    [{cid:3d}] {component_dict[cid]["name"]:40s} \u2014 {cnt}x')

print(f'\n  Sample GT (first 8):')
for p in dataset[:8]:
    names = [component_dict[cid]['name'] for cid in p['ground_truth']]
    print(f'    P{p["project_id"]:3d} | scored={p["ground_truth"]} | {names}')

# ── 9. Audit: flag projects with >8 scored components ─────────────────────────
oversize = [p for p in dataset if len(p['ground_truth']) > 8]
if oversize:
    print(f'\n  \u26a0\ufe0f {len(oversize)} projects have >8 scored components (inspect manually):')
    for p in oversize:
        names = [component_dict[c]['name'] for c in p['ground_truth']]
        print(f'    P{p["project_id"]:3d} ({len(p["ground_truth"])} comps): {names}')
else:
    print(f'\n  \u2705 No projects exceed 8 scored components')


In [ ]:
# ── CELL 5: CODE SIGNAL EXTRACTION ───────────────────────────────────────────
SIGNAL_PATTERNS = [
    (r'#include\s*[<"](\S+?)[">]',        lambda m: f'Library include: {m.group(1)}'),
    (r'analogRead\s*\(',                   lambda _: 'Uses analogRead (analog sensor/input)'),
    (r'analogWrite\s*\(',                  lambda _: 'Uses analogWrite (PWM output)'),
    (r'digitalRead\s*\(',                  lambda _: 'Uses digitalRead (digital input)'),
    (r'digitalWrite\s*\(',                 lambda _: 'Uses digitalWrite (digital output)'),
    (r'tone\s*\(',                         lambda _: 'Uses tone() (buzzer/speaker)'),
    (r'\bServo\b',                         lambda _: 'Uses Servo library/object'),
    (r'\bLiquidCrystal\b|\bLCD\b',        lambda _: 'Uses LCD display'),
    (r'\bDHT\b',                           lambda _: 'Uses DHT temperature/humidity sensor'),
    (r'\bWire\b|\.begin\s*\(\s*\)',       lambda _: 'Uses I2C (Wire library)'),
    (r'\bSPI\b',                           lambda _: 'Uses SPI bus'),
    (r'\bSoftwareSerial\b|Serial\.begin', lambda _: 'Uses Serial/UART communication'),
    (r'\bstepper\b|\bStepper\b',          lambda _: 'Uses Stepper motor'),
    (r'\bIRremote\b|\bIRrecv\b',          lambda _: 'Uses IR remote receiver'),
    (r'\bEthernet\b|\bWiFi\b',            lambda _: 'Uses network module (Ethernet/WiFi)'),
    (r'\bSD\.begin\b',                     lambda _: 'Uses SD card module'),
    (r'\bpulseIn\b',                       lambda _: 'Uses pulseIn (ultrasonic/timing)'),
    (r'\bNeoPixel\b|\bFastLED\b',         lambda _: 'Uses addressable LED strip'),
]

def extract_code_signals(code: str) -> str:
    """Return human-readable signals extracted from Arduino .ino code."""
    if not code.strip():
        return 'No code available.'
    seen   : Set[str] = set()
    signals: List[str] = []
    for pattern, formatter in SIGNAL_PATTERNS:
        for match in re.finditer(pattern, code, re.IGNORECASE):
            signal = formatter(match)
            if signal not in seen:
                seen.add(signal)
                signals.append(f'  • {signal}')
    return '\n'.join(signals) if signals else '  • No recognisable hardware signals detected.'

# Quick test — load code from code_path (projects store path, not inline code)
_base = Path('..')  # notebook is in notebooks/, data in project root
_sample_path = _base / dataset[0].get('code_path', '').lstrip('./')
sample_code = _sample_path.read_text() if _sample_path.exists() else ''
print('=== CODE SIGNALS SAMPLE ===')
print(extract_code_signals(sample_code))

# 🔨 Task 6: Code Synthesis
> **Measures:** Physical quality of the generated `.ino` code across multiple dimensions:

| Metric | Description |
|--------|-------------|
| **Compilation** | Does the code compile with `arduino-cli` for the target board? |
| **Library coverage** | Are all needed `#include` statements present? |
| **Pin correctness** | Are pin assignments sensible for the board? |
| **Structural completeness** | Has valid `setup()` + `loop()`? |
| **No hallucinations** | Are all component IDs from the canonical dictionary? |

**Method:** `run_pipeline` Stage 2 — `SYSTEM_PROMPT_SYNTHESIS` → raw `.ino` output → `arduino-cli compile` → pass/fail.

**Status:** ❌ Not started — requires `arduino-cli` integration.

In [ ]:
# ── CELL 6: CONTEXT BUILDER ───────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compute TF-IDF similarity matrix on the dataset descriptions
try:
    _descriptions = [str(p.get('description', '')) for p in dataset]
    _vectorizer = TfidfVectorizer(stop_words='english')
    _tfidf_matrix = _vectorizer.fit_transform(_descriptions)
    _similarity_matrix = cosine_similarity(_tfidf_matrix)
    print("TF-IDF RAG Similarity Matrix Computed!")
except Exception as e:
    print("Warning: Could not compute TF-IDF matrix (is dataset completely loaded?)", e)

MAX_DESC_CHARS = 600
MAX_CODE_CHARS = 1500
MAX_RAG_DESC   = 200
MAX_RAG_CODE   = 600
MAX_RAG_K      = 3    # reduced from 5

def get_top_k_similar_projects(current_project: Dict, k=5) -> List[Dict]:
    try:
        pid = current_project['project_id']
        idx = next((i for i, p in enumerate(dataset) if p['project_id'] == pid), -1)
        if idx == -1: return []
        
        sims = _similarity_matrix[idx]
        top_indices = np.argsort(sims)[::-1]
        
        results = []
        for i in top_indices:
            if i == idx: continue  # skip self
            if len(results) >= k: break
            results.append(dataset[i])
        return results
    except:
        return []

def build_prompt_context(project: Dict, component_ctx: str) -> str:
    """Assemble the full LLM prompt context for a single project."""
    description = project['description'].strip() or 'Not provided.'
    description = (description[:MAX_DESC_CHARS] + '\n... [truncated]') if len(description) > MAX_DESC_CHARS else description
    # Load code from code_path (projects store path, not inline code)
    _cp = project.get('code_path', '')
    _base_path = Path('..') if Path('../data').exists() else Path('.')
    _code_file = (_base_path / _cp.lstrip('./')) if _cp else None
    code = _code_file.read_text() if (_code_file and _code_file.exists()) else project.get('code', '')
    code_snippet= (code[:MAX_CODE_CHARS] + '\n... [truncated]') if len(code) > MAX_CODE_CHARS else code
    code_snippet= code_snippet.strip() or 'Not provided.'
    signals     = extract_code_signals(code)
    
    # RAG Retrieval — capped at 3 examples, tighter per-example limits
    top_similar = get_top_k_similar_projects(project, k=MAX_RAG_K)
    rag_context = f"\n\n=== SIMILAR RAG EXAMPLES (TOP {MAX_RAG_K}) ===\n"
    if not top_similar:
        rag_context += "No RAG examples found.\n"
    for i, p in enumerate(top_similar, 1):
        s_desc = str(p.get('description', ''))[:MAX_RAG_DESC]
        # Load code from code_path if available, else fall back to inline 'code'
        _cp = p.get('code_path', '')
        _base_path = Path('..') if Path('../data').exists() else Path('.')
        _code_file = (_base_path / _cp.lstrip('./')) if _cp else None
        s_code = _code_file.read_text()[:MAX_RAG_CODE] if (_code_file and _code_file.exists()) else str(p.get('code', ''))[:MAX_RAG_CODE]
        rag_context += f"--- Example {i} ---\nDescription: {s_desc}\nCode:\n```cpp\n{s_code}\n```\n\n"

    prompt = f"""=== PROJECT DESCRIPTION ===
{description}

=== RAW ARDUINO CODE ===
{code_snippet}

=== CODE SIGNALS (extracted) ===
{signals}

=== {component_ctx}
{rag_context}
"""
    return prompt.strip()


sample_context = build_prompt_context(dataset[0], component_context)
print(sample_context[:1200], '\n...')
def build_code_prompt(description, component_names, rag_examples):
    comp_list_str = "\n".join(f"  - {name}" for name in component_names)
    
    rag_str = ""
    for i, p in enumerate(rag_examples[:3], 1):  # 3 examples is enough for code
        # Load code from code_path (projects store path, not inline code)
        _cp = p.get('code_path', '')
        _base_path = Path('..') if Path('../data').exists() else Path('.')
        _code_file = (_base_path / _cp.lstrip('./')) if _cp else None
        s_code = _code_file.read_text()[:1500] if (_code_file and _code_file.exists()) else str(p.get('code', ''))[:1500]
        
        # Format the RAG block
        rag_str += f"--- Example {i} ---\n"
        rag_str += f"Description: {str(p.get('description',''))[:300]}\n"
        rag_str += f"Code:\n```cpp\n{s_code}\n```\n\n"

    return f"""PROJECT DESCRIPTION:
{description}

DETECTED COMPONENTS:
{comp_list_str}

SIMILAR RAG EXAMPLES:
{rag_str}

Now write the complete Arduino .ino code:"""


## 🔥 CELL 7 — STRICT MASTER PROMPT TEMPLATE

The following prompt is injected verbatim into every LLM call.

```
You are a STRICT Arduino project analyzer.

Use ALL signals:
1. Description
2. Code
3. Code signals
4. Canonical component list

RULES:
- Only use components from the canonical list (use their exact integer IDs)
- No hallucination — never invent component IDs
- Prefer recall over precision (include if reasonably likely)
- Code evidence outweighs description evidence
- If unsure, include with confidence = 'low'

OUTPUT FORMAT (strict JSON, no extra text):
{
  "components": [
    {
      "component_id": <int>,
      "confidence": "high|medium|low",
      "evidence": "code|description|both"
    }
  ]
}
```


SYSTEM_PROMPT_INTENT = """
You are an Arduino Logic Architect. 
Your task is to convert a project description and component list into a high-level LOGIC FLOW.

OUTPUT FORMAT (strict JSON):
{
  "logic_steps": [
    {"step": 1, "description": "Initialize sensors and serial communication"},
    {"step": 2, "description": "Read distance from Ultrasonic sensor"},
    {"step": 3, "description": "If distance < 20cm, open lid via Servo"},
    {"step": 4, "description": "Wait 2 seconds, then close lid"}
  ],
  "conditions": ["Distance < 20cm triggered"],
  "inputs": ["Ultrasonic Echo"],
  "outputs": ["Servo Signal", "Ultrasonic Trigger"]
}
"""

SYSTEM_PROMPT_MAPPING = """
You are a Physical Hardware Mapper for Arduino.
Given a list of components and their dictionary metadata (Pin Type), assign them to specific pins on the Arduino Uno.

RULES:
- Use internal pullups for buttons if possible.
- I2C components MUST use A4 (SDA) and A5 (SCL).
- PWM components must use pins 3, 5, 6, 9, 10, or 11.
- DO NOT overlap pins.

OUTPUT FORMAT (strict JSON):
{
  "assignments": [
    {"component": "Ultrasonic Sensor", "pins": {"trig": 5, "echo": 6}, "type": "Digital"},
    {"component": "Servo Motor", "pins": {"signal": 9}, "type": "PWM"}
  ]
}
"""

SYSTEM_PROMPT_PLANNING = """
You are an Arduino Senior Developer.
Create a structured coding plan (Pseudo-header) based on the logic flow and pin mapping.

OUTPUT FORMAT (strict JSON):
{
  "libraries": ["Servo.h"],
  "global_vars": [
    {"name": "servo", "type": "Servo", "description": "Servo object"},
    {"name": "dist", "type": "long", "description": "Distance tracker"}
  ],
  "setup_logic": ["Attach servo to pin 9", "Set trigPin as OUTPUT"],
  "loop_logic": ["Trigger pulse", "Calculate distance", "Determine servo motor position"]
}
"""


In [ ]:
from openai import OpenAI

# ── CELL 8: LLM CLIENT + CALL FUNCTION ─────────────────────────────────────────
import time as _time
import random

SYSTEM_PROMPT_EXTRACTION = """You are a STRICT Arduino project analyzer.

Use ALL signals:
1. Project description
2. Raw Arduino code
3. Extracted code signals
4. Canonical component list
5. Similar RAG examples

RULES:
- Only use component IDs from the canonical list
- Never invent component IDs
- Prefer recall over precision (include if reasonably likely)
- Code evidence outweighs description evidence
- If unsure, include with confidence = 'low'

OUTPUT FORMAT (strict JSON, no extra text):
{
  "components": [
    {
      "component_id": <int>,
      "confidence": "high|medium|low",
      "evidence": "code|description|both"
    }
  ]
}"""


SYSTEM_PROMPT_SYNTHESIS = """You are an expert Arduino firmware engineer.

You will be given:
1. A project description
2. A list of hardware components (with their names)
3. Similar RAG example projects (description + code)

Your task is to write complete, working Arduino .ino code for this project.

RULES:
- Include all necessary #include statements for the given components
- Implement void setup() and void loop() correctly
- Use realistic pin assignments (comment them clearly)
- Do NOT copy code from the RAG examples — use them only as style reference
- Do NOT include any explanation, markdown, or commentary outside the code
- Output raw .ino code only, starting with the first #include or void setup()"""
# ── MODULE 1: Component Extraction Agent Prompt ───────────────────────────────
SYSTEM_PROMPT_MODULE1 = """You are a hardware component extraction specialist for Arduino and embedded systems projects.

Your task is to extract ALL hardware components from a natural language project description. A component is any physical electronic part: sensors, actuators, modules, displays, motors, LEDs, relays, or communication devices. Do NOT include boards (Arduino, ESP32, etc.), wires, resistors, capacitors, or breadboards.

Rules:
- Extract components explicitly named AND components implied by function (e.g. "measures temperature" → temperature sensor)
- Normalise synonyms to canonical names (e.g. "HC-SR04" → "Ultrasonic Sensor", "SG90" → "Servo Motor")
- If a library name is provided (e.g. "Servo.h", "DHT.h"), infer the associated component
- Output ONLY a JSON array. No explanation, no markdown, no preamble.

You are given a fixed component vocabulary.

Return ONLY components that exist in this list:
<INSERT COMPONENT LIST HERE>

If a detected component is not in the list, map it to the closest valid one.
If no reasonable match exists, ignore it.

Output format:
[
  {
    "canonical_name": "Ultrasonic Sensor",
    "raw_mention": "HC-SR04",
    "confidence": "high"
  }
]

Confidence levels:
- "high"   → explicitly named or strong library signal
- "medium" → inferred from function description
- "low"    → inferred from domain context only

--- EXAMPLES ---

Example 1:
Description: "Smart dustbin that uses an ultrasonic sensor to detect nearby objects and opens the lid using a servo motor."
Libraries: Servo.h
Output:
[
  {"canonical_name": "Ultrasonic Sensor", "raw_mention": "ultrasonic sensor", "confidence": "high"},
  {"canonical_name": "Servo Motor",       "raw_mention": "servo motor",       "confidence": "high"}
]

Example 2:
Description: "Plant watering system that monitors soil moisture and controls a pump via Blynk, with LED matrix feedback."
Libraries: BlynkSimpleWifi, Arduino_LED_Matrix, EEPROM
Output:
[
  {"canonical_name": "Soil Moisture Sensor", "raw_mention": "soil moisture",   "confidence": "high"},
  {"canonical_name": "Relay Module",         "raw_mention": "controls a pump", "confidence": "medium"},
  {"canonical_name": "LED Matrix",           "raw_mention": "LED matrix",      "confidence": "high"}
]

Example 3:
Description: "Line follower robot using infrared sensors and DC motors controlled by an L293D motor driver."
Libraries: None
Output:
[
  {"canonical_name": "IR Sensor",          "raw_mention": "infrared sensors", "confidence": "high"},
  {"canonical_name": "DC Motor",           "raw_mention": "DC motors",        "confidence": "high"},
  {"canonical_name": "L293D Motor Driver", "raw_mention": "L293D",            "confidence": "high"}
]

--- END EXAMPLES ---"""

USER_PROMPT_MODULE1_TEMPLATE = """Extract all hardware components from the following Arduino project.

Project description:
{description}

Libraries used (if known):
{libraries}

Return ONLY the JSON array."""


# ── MODULE 1: Dictionary Lookup ────────────────────────────────────────────────
import difflib

def module1_lookup(canonical_name: str, df_components, threshold: float = 0.6) -> Optional[int]:
    """
    Map a canonical component name (from LLM) to a Component_Id via fuzzy match.
    Returns the best-matching Component_Id or None if below threshold.
    """
    name_lower = canonical_name.lower().strip()
    best_score = 0.0
    best_id = None

    for _, row in df_components.iterrows():
        comp_name = str(row["Component name"]).lower()
        score = difflib.SequenceMatcher(None, name_lower, comp_name).ratio()
        if score > best_score:
            best_score = score
            best_id = int(row["Component_Id"])

    return best_id if best_score >= threshold else None


def run_module1(project: dict, df_components, model_client, threshold: float = 0.6) -> dict:
    """
    Module 1: LLM extracts canonical component names → fuzzy-map to dictionary IDs.
    Returns predicted_ids (set), raw_extractions (list), and per-item confidence.
    """
    desc = str(project.get("description", ""))
    libs = project.get("ground_truth_libraries", [])
    lib_str = ", ".join(libs) if isinstance(libs, list) else str(libs)

    user_prompt = USER_PROMPT_MODULE1_TEMPLATE.format(
        description=desc,
        libraries=lib_str or "None provided"
    )

    raw_output, latency = call_llm(
        user_prompt,
        model=model_client,
        system_prompt=SYSTEM_PROMPT_MODULE1
    )

    # raw_output is already parsed JSON from call_llm
    extractions = raw_output if isinstance(raw_output, list) else []

    predicted_ids = set()
    resolved = []

    for item in extractions:
        if not isinstance(item, dict):
            continue
        cname = item.get("canonical_name", "")
        conf  = item.get("confidence", "low")
        cid   = module1_lookup(cname, df_components, threshold=threshold)

        resolved.append({
            "canonical_name": cname,
            "raw_mention":    item.get("raw_mention", ""),
            "confidence":     conf,
            "resolved_id":    cid,
        })

        if cid is not None:
            predicted_ids.add(cid)

    return {
        "predicted_ids": predicted_ids,
        "extractions":   resolved,
        "latency":       latency,
    }


# ── MODULE 1: Component Extraction Agent Prompt ───────────────────────────────
SYSTEM_PROMPT_MODULE1 = """You are a hardware component extraction specialist for Arduino and embedded systems projects.

Your task is to extract ALL hardware components from a natural language project description. A component is any physical electronic part: sensors, actuators, modules, displays, motors, LEDs, relays, or communication devices. Do NOT include boards (Arduino, ESP32, etc.), wires, resistors, capacitors, or breadboards.

Rules:
- Extract components explicitly named AND components implied by function (e.g. "measures temperature" → temperature sensor)
- Normalise synonyms to canonical names (e.g. "HC-SR04" → "Ultrasonic Sensor", "SG90" → "Servo Motor")
- If a library name is provided (e.g. "Servo.h", "DHT.h"), infer the associated component
- Output ONLY a JSON array. No explanation, no markdown, no preamble.

Output format:
[
  {
    "canonical_name": "Ultrasonic Sensor",
    "raw_mention": "HC-SR04",
    "confidence": "high"
  }
]

Confidence levels:
- "high"   → explicitly named or strong library signal
- "medium" → inferred from function description
- "low"    → inferred from domain context only"""

USER_PROMPT_MODULE1_TEMPLATE = """Extract all hardware components from the following Arduino project.

Project description:
{description}

Libraries used (if known):
{libraries}

Return ONLY the JSON array."""


# ── MODULE 1: Dictionary Lookup ────────────────────────────────────────────────
import difflib

def module1_lookup(canonical_name: str, df_components, threshold: float = 0.6) -> Optional[int]:
    """
    Map a canonical component name (from LLM) to a Component_Id via fuzzy match.
    Returns the best-matching Component_Id or None if below threshold.
    """
    name_lower = canonical_name.lower().strip()
    best_score = 0.0
    best_id = None

    for _, row in df_components.iterrows():
        comp_name = str(row["Component name"]).lower()
        score = difflib.SequenceMatcher(None, name_lower, comp_name).ratio()
        if score > best_score:
            best_score = score
            best_id = int(row["Component_Id"])

    return best_id if best_score >= threshold else None


def run_module1(project: dict, df_components, model_client, threshold: float = 0.6) -> dict:
    """
    Module 1: LLM extracts canonical component names → fuzzy-map to dictionary IDs.
    Returns predicted_ids (set), raw_extractions (list), and per-item confidence.
    """
    desc = str(project.get("description", ""))
    libs = project.get("ground_truth_libraries", [])
    lib_str = ", ".join(libs) if isinstance(libs, list) else str(libs)

    user_prompt = USER_PROMPT_MODULE1_TEMPLATE.format(
        description=desc,
        libraries=lib_str or "None provided"
    )

    raw_output, latency = call_llm(
        user_prompt,
        model=model_client,
        system_prompt=SYSTEM_PROMPT_MODULE1
    )

    # raw_output is already parsed JSON from call_llm
    extractions = raw_output if isinstance(raw_output, list) else []

    predicted_ids = set()
    resolved = []

    for item in extractions:
        if not isinstance(item, dict):
            continue
        cname = item.get("canonical_name", "")
        conf  = item.get("confidence", "low")
        cid   = module1_lookup(cname, df_components, threshold=threshold)

        resolved.append({
            "canonical_name": cname,
            "raw_mention":    item.get("raw_mention", ""),
            "confidence":     conf,
            "resolved_id":    cid,
        })

        if cid is not None:
            predicted_ids.add(cid)

    return {
        "predicted_ids": predicted_ids,
        "extractions":   resolved,
        "latency":       latency,
    }


# ── Model registry ─────────────────────────────────────────────────────────────
MODELS = {
    'llama3_3':   ('nvidia', 'meta/llama-3.3-70b-instruct',                  'NVIDIA_API_KEY_LLAMA33'),
    'mistral':    ('nvidia', 'mistralai/mistral-large-3-675b-instruct-2512', 'NVIDIA_API_KEY_MISTRAL'),
    'phi_medium': ('nvidia', 'microsoft/phi-3-medium-128k-instruct',         'NVIDIA_API_KEY_PHI3MEDIUM'),
    'gemma_27b':  ('nvidia', 'google/gemma-2-27b-it',                        'NVIDIA_API_KEY_GEMMA'),
    'phi_mini':   ('nvidia', 'microsoft/phi-3-mini-128k-instruct',           'NVIDIA_API_KEY_PHI3MINI'),
}

NVIDIA_BASE_URL = 'https://integrate.api.nvidia.com/v1'

# ── Tunables ───────────────────────────────────────────────────────────────────
API_CALL_DELAY  = 3.0     # seconds between calls (rate-limit guard)
API_TIMEOUT     = 30      # seconds per API call
MAX_RETRIES     = 5       # retry on transient failures
MAX_PROMPT_CHARS = 32000  # hard cap on prompt length


class LLMClient:
    """OpenAI-compatible client for NVIDIA-hosted models."""

    def __init__(self, model_key, temperature=0):
        if model_key not in MODELS:
            raise ValueError(f'Unknown model_key: {model_key!r}. Valid: {list(MODELS.keys())}')

        self.model_key   = model_key
        self.provider, self.model_name, api_key_name = MODELS[model_key]
        self.temperature = temperature
        self.total_cost  = 0.0
        self.call_log    = []

        api_key = os.getenv(api_key_name)
        if not api_key:
            raise ValueError(f'Missing API key: set {api_key_name} in .env')

        self.client = OpenAI(
            base_url=NVIDIA_BASE_URL,
            api_key=api_key
        )
        logger.info(f'LLMClient ready: {model_key} ({self.model_name})')

    def generate(self, prompt: str, system_prompt: str = SYSTEM_PROMPT_EXTRACTION) -> Tuple[str, float]:
        """
        Send prompt → return (raw_text, inference_latency_seconds).
        - Latency = pure API call time
        - Retries up to MAX_RETRIES on transient failures
        - Timeout protection per call
        - Returns ('', 0.0) on final failure
        """
        last_exc = None
        for attempt in range(MAX_RETRIES):
            try:
                _time.sleep(API_CALL_DELAY)
                t0 = _time.time()
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages = (
                        [{'role': 'user', 'content': system_prompt + "\n\n" + prompt}]
                        if self.model_key == 'gemma_27b' else
                        [
                            {'role': 'system', 'content': system_prompt},
                            {'role': 'user',   'content': prompt},
                        ]
                    ),
                    temperature=self.temperature,
                    max_tokens=512,
                    timeout=API_TIMEOUT,
                )
                latency = _time.time() - t0
                text = response.choices[0].message.content.strip()

                # Track token usage if available
                usage = getattr(response, 'usage', None)
                log_entry = {
                    'latency': round(latency, 3),
                    'prompt_len': len(prompt),
                    'response_len': len(text),
                    'attempt': attempt + 1,
                    'success': True,
                }
                if usage:
                    log_entry['tokens_prompt']     = getattr(usage, 'prompt_tokens', None)
                    log_entry['tokens_completion']  = getattr(usage, 'completion_tokens', None)
                self.call_log.append(log_entry)

                return text, latency

            except Exception as exc:
                last_exc = exc
                if attempt < MAX_RETRIES - 1:
                    wait = 0.5 * (attempt + 1)
                    if '429' in str(exc):
                        retry_after = getattr(getattr(exc, 'response', None), 'headers', {}).get('Retry-After')
                        if retry_after:
                            wait = float(retry_after) + random.uniform(0.5, 2.0)
                        else:
                            wait = min(120.0, 15.0 * (2 ** attempt) + random.uniform(1, 3))
                    logger.warning(f'{self.model_key} attempt {attempt+1}/{MAX_RETRIES} failed: {exc}. '
                                   f'Retrying in {wait:.1f}s...')
                    _time.sleep(wait)

        # All retries exhausted
        logger.warning(f'LLM call FAILED after {MAX_RETRIES} retries ({self.model_key}): {last_exc}')
        self.call_log.append({'latency': 0.0, 'success': False, 'error': str(last_exc)[:100]})
        return '', 0.0


def call_llm(prompt: str, model = 'stub', system_prompt=SYSTEM_PROMPT_EXTRACTION) -> Tuple[dict, float]:
    """
    Unified LLM call. Returns (parsed_dict, inference_latency_s).
    model must be 'stub' or a pre-initialised LLMClient instance.
    """
    if model == 'stub':
        return {'components': []}, 0.0

    if not isinstance(model, LLMClient):
        raise TypeError(f'model must be "stub" or LLMClient instance, got {type(model).__name__}. '
                        f'Pre-initialise: client = LLMClient("{model}")')

    # Hard cap prompt size
    if len(prompt) > MAX_PROMPT_CHARS:
        logger.warning(f'Prompt truncated: {len(prompt)} → {MAX_PROMPT_CHARS} chars')
        prompt = prompt[:MAX_PROMPT_CHARS - 200] + '\n\n[TRUNCATED]'

    raw, latency = model.generate(prompt, system_prompt=system_prompt)

    if not raw:
        return {'components': []}, latency


    # ── Robust Parsing ──
    import json, re

    text = raw.strip()
    parsed_ids = []

    # Remove markdown fences
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\n?", "", text)
        text = re.sub(r"\n?```$", "", text).strip()

    # ✅ 1. Try full JSON FIRST (THIS IS THE KEY FIX)
    try:
        data = json.loads(text)

        if isinstance(data, list):
            parsed_ids = data

        elif isinstance(data, dict) and "components" in data:
            parsed_ids = [
                c.get("component_id")
                for c in data["components"]
                if isinstance(c, dict) and "component_id" in c
            ]

    except Exception:
        pass

    # ✅ 2. Fallback regex ONLY if still empty
    if not parsed_ids:
        match = re.search(r"\[\s*\d+(?:\s*,\s*\d+)*\s*\]", text)
        if match:
            try:
                parsed_ids = json.loads(match.group(0))
            except:
                pass

    # Normalize
    final_ids = []
    for x in parsed_ids:
        try:
            final_ids.append(int(x))
        except:
            pass

    out_dict = {}
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            out_dict = data
    except Exception:
        pass
        
    out_dict['components'] = [{'component_id': cid} for cid in final_ids]
    
    
    # Attempt to sniff markdown code blocks directly if not in JSON
    if 'code' not in out_dict and 'arduino_code' not in out_dict:
        code_blocks = re.findall(r"```(?:cpp|c\+\+|c|arduino|\n)?\s*(.*?)```", raw, re.DOTALL | re.IGNORECASE)
        for block in code_blocks:
            if 'void setup' in block or 'void loop' in block or 'include' in block:
                out_dict['code'] = block.strip()
                break
                
    out_dict['raw_response'] = raw


    # 🚨 Debug check
    if not final_ids:
        print("DEBUG RAW OUTPUT:\n", text[:500])

    return out_dict, latency



def call_llm_raw(prompt: str, model, system_prompt: str) -> Tuple[str, float]:
    """For free-form text generation (code synthesis) — no JSON parsing."""
    if model == 'stub':
        return '', 0.0
    if not isinstance(model, LLMClient):
        raise TypeError(f'model must be "stub" or LLMClient instance')

    if len(prompt) > MAX_PROMPT_CHARS:
        prompt = prompt[:MAX_PROMPT_CHARS - 200] + '\n\n[TRUNCATED]'

    raw, latency = model.generate(prompt, system_prompt=system_prompt)
    
    # Strip markdown fences if model wrapped output anyway
    import re
    text = raw.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:cpp|c\+\+|c|arduino)?\n?', '', text)
        text = re.sub(r'\n?```$', '', text).strip()
    
    return text, latency

ACTIVE_MODEL = 'stub'
print(f'Active model: {ACTIVE_MODEL}')
print(f'Available models: {list(MODELS.keys())}')
print(f'Config: timeout={API_TIMEOUT}s, retries={MAX_RETRIES}, delay={API_CALL_DELAY}s, max_prompt={MAX_PROMPT_CHARS}')


In [ ]:
# ── CELL 9: NORMALIZATION ─────────────────────────────────────────────────────
VALID_IDS: Set[int] = set(component_dict.keys())


def normalize_predictions(llm_output: dict) -> Set[int]:
    """
    Extract component IDs from LLM JSON output.
    - Deduplicates
    - Validates against known component IDs
    - Returns Set[int]
    """
    if not isinstance(llm_output, dict):
        return set()
    components = llm_output.get('components', [])
    if not isinstance(components, list):
        return set()

    ids: Set[int] = set()
    for item in components:
        if not isinstance(item, dict):
            continue
        cid = item.get('component_id')
        try:
            cid = int(cid)
        except (TypeError, ValueError):
            logger.debug(f'Invalid component_id: {cid}')
            continue
        if cid in VALID_IDS:
            ids.add(cid)
        else:
            logger.debug(f'Hallucinated component_id (not in dict): {cid}')
    return ids


# Quick test
test_out = {'components': [{'component_id': 1, 'confidence': 'high', 'evidence': 'code'},
                            {'component_id': 9999, 'confidence': 'low', 'evidence': 'description'}]}
print('Normalised IDs:', normalize_predictions(test_out))

In [ ]:
# ── CELL 10: METRICS ENGINE ───────────────────────────────────────────────────
def compute_basic_metrics(pred: Set[int], gt: Set[int]) -> Dict:
    """
    Exact-match Precision, Recall, F1.
    - Both empty  → P=R=F1=1.0
    - One empty   → all 0.0
    """
    if len(pred) == 0 and len(gt) == 0:
        return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'tp': 0, 'fp': 0, 'fn': 0}
    if len(pred) == 0 or len(gt) == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0,
                'tp': 0, 'fp': len(pred), 'fn': len(gt)}

    tp = len(pred & gt)
    fp = len(pred - gt)
    fn = len(gt  - pred)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return {'precision': round(precision, 4), 'recall': round(recall, 4),
            'f1': round(f1, 4), 'tp': tp, 'fp': fp, 'fn': fn}


# Sanity checks
assert compute_basic_metrics(set(), set())['f1'] == 1.0
assert compute_basic_metrics({1}, set())['f1']   == 0.0
assert compute_basic_metrics({1, 2}, {1, 2})['f1'] == 1.0
print(' Metric engine sanity checks passed')

In [ ]:
# ── CELL 11: RELAXED / ALIAS-BASED MATCHING ───────────────────────────────────
STOPWORDS = {'the', 'a', 'an', 'and', 'or', 'with', 'for', 'of', 'to', 'in', 'is', 'module', 'sensor'}


def tokenize(text: str) -> Set[str]:
    """Lowercase, split, remove stopwords."""
    tokens = re.findall(r'[a-z0-9]+', text.lower())
    return {t for t in tokens if t not in STOPWORDS}


def name_token_overlap(name_a: str, name_b: str) -> float:
    """Jaccard token overlap between two component name strings."""
    ta, tb = tokenize(name_a), tokenize(name_b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def relaxed_match(pred_id: int, gt_id: int, comp_dict: Dict,
                  alias_map: Dict, threshold: float = 0.5) -> bool:
    """Return True if pred_id is a relaxed match for gt_id (via alias or token overlap)."""
    if pred_id == gt_id:
        return True
    pred_entry = comp_dict.get(pred_id)
    gt_entry   = comp_dict.get(gt_id)
    if not pred_entry or not gt_entry:
        return False
    # Alias-based
    gt_aliases = {gt_entry['name'].lower()} | {a.lower() for a in gt_entry['aliases']}
    pred_all   = {pred_entry['name'].lower()} | {a.lower() for a in pred_entry['aliases']}
    if pred_all & gt_aliases:
        return True
    # Token overlap
    return name_token_overlap(pred_entry['name'], gt_entry['name']) >= threshold


def compute_relaxed_metrics(pred: Set[int], gt: Set[int],
                             comp_dict: Dict, alias_map: Dict) -> Dict:
    """Relaxed P/R/F1 using alias + token matching."""
    if len(pred) == 0 and len(gt) == 0:
        return {'relaxed_precision': 1.0, 'relaxed_recall': 1.0, 'relaxed_f1': 1.0}
    if len(pred) == 0 or len(gt) == 0:
        return {'relaxed_precision': 0.0, 'relaxed_recall': 0.0, 'relaxed_f1': 0.0}

    tp_pred  = sum(1 for p in pred if any(relaxed_match(p, g, comp_dict, alias_map) for g in gt))
    tp_gt    = sum(1 for g in gt  if any(relaxed_match(p, g, comp_dict, alias_map) for p in pred))
    precision = tp_pred / len(pred)
    recall    = tp_gt   / len(gt)
    f1        = (2 * precision * recall / (precision + recall)
                 if precision + recall > 0 else 0.0)
    return {'relaxed_precision': round(precision, 4),
            'relaxed_recall'   : round(recall,    4),
            'relaxed_f1'       : round(f1,        4)}


print(' Relaxed matching functions defined')

In [ ]:
# ── CELL 12: MODULAR PIPELINE (6-STAGE REASONING) ────────────────────────────

def run_pipeline(project: Dict,
                 hw_dict: HardwareDictionary,
                 comp_ctx: str,
                 model = 'stub',
                 ablation: str = 'full') -> Dict:
    """
    A 6-Stage modular pipeline for Arduino code generation.
    1. Extraction (Module 1)
    2. Intent Understanding (Module 3A)
    3. Hardware Mapping (Module 3B)
    4. Code Planning (Module 3C)
    5. Code Synthesis (Stage 2)
    6. Compilation (Future Stage 3)
    """
    pid = project['project_id']
    total_latency = 0.0
    stages = {}

    try:
        # Create a deep copy for ablation
        ablated = dict(project)
        if ablation == 'desc_only': ablated['code_path'] = ''
        elif ablation == 'code_only': ablated['description'] = ''

        # ── STAGE 1: EXTRACTION (Module 1) ───────────────────────────────────
        m1_res = run_module1(ablated, hw_dict.df, model, threshold=0.6)
        stages['extraction'] = m1_res
        total_latency += m1_res.get('latency', 0.0)
        
        pred_ids = m1_res.get('predicted_ids', [])
        component_names = [hw_dict.lookup(cid)['name'] for cid in pred_ids]

        # ── STAGE 2: INTENT (Module 3A) ──────────────────────────────────────
        intent_user_prompt = f"Project: {ablated.get('description', '')}\nComponents: {', '.join(component_names)}"
        intent_res, lat_intent = call_llm(
            intent_user_prompt, 
            model=model, 
            system_prompt=SYSTEM_PROMPT_INTENT
        )
        stages['intent'] = intent_res
        total_latency += lat_intent

        # ── STAGE 3: MAPPING (Module 3B) ─────────────────────────────────────
        mapping_ctx = "\n".join([
            f"  - {hw_dict.lookup(cid)['name']} | Pins: {hw_dict.lookup(cid)['pin_type']}"
            for cid in pred_ids
        ])
        mapping_user_prompt = f"Project description: {ablated.get('description', '')}\nHardware context:\n{mapping_ctx}"
        mapping_res, lat_mapping = call_llm(
            mapping_user_prompt, 
            model=model, 
            system_prompt=SYSTEM_PROMPT_MAPPING
        )
        stages['mapping'] = mapping_res
        total_latency += lat_mapping

        # ── STAGE 4: PLANNING (Module 3C) ────────────────────────────────────
        planning_user_prompt = f"Logic Flow: {intent_res}\nPin Mapping: {mapping_res}"
        planning_res, lat_planning = call_llm(
            planning_user_prompt, 
            model=model, 
            system_prompt=SYSTEM_PROMPT_PLANNING
        )
        stages['planning'] = planning_res
        total_latency += lat_planning

        # ── STAGE 5: SYNTHESIS (Stage 2) ─────────────────────────────────────
        top_similar = get_top_k_similar_projects(project, k=3)
        code_prompt = build_code_prompt(ablated.get('description', ''), component_names, top_similar)
        synthesis_res, lat_synthesis = call_llm(
            code_prompt, 
            model=model, 
            system_prompt=SYSTEM_PROMPT_SYNTHESIS
        )
        # Note: synth_res is the actual code string
        stages['synthesis'] = synthesis_res
        total_latency += lat_synthesis

        # ── METRICS (Task 1: Component Extraction) ───────────────────────────
        gt = set(project.get('ground_truth', []))
        metrics = calculate_metrics(list(pred_ids), list(gt))
        relaxed = calculate_relaxed_metrics(list(pred_ids), list(gt), hw_dict.comp_dict, hw_dict.alias_to_id)
        metrics.update(relaxed)

        return {
            'project_id'    : pid,
            'ablation'      : ablation,
            'predictions'   : list(pred_ids),
            'ground_truth'  : list(gt),
            'stages'        : stages,
            'metrics'       : metrics,
            'latency'       : round(total_latency, 3),
            'error'         : None,
        }

    except Exception as exc:
        import traceback
        logger.error(f'Pipeline error for project {pid}: {exc}\n{traceback.format_exc()}')
        return {
            'project_id'    : pid,
            'ablation'      : ablation,
            'predictions'   : [],
            'ground_truth'  : list(project.get('ground_truth', [])),
            'stages'        : stages,
            'metrics'       : {'precision': 0, 'recall': 0, 'f1': 0, 'tp': 0, 'fp': 0, 'fn': 0,
                               'relaxed_precision': 0, 'relaxed_recall': 0, 'relaxed_f1': 0},
            'latency'       : total_latency,
            'error'         : str(exc),
        }

In [ ]:
# # ── CELL 13: RUN ALL PROJECTS (multi-model, incremental saves) ────────────────
# from tqdm import tqdm
# from datetime import datetime
# import time
# from openai import OpenAI

# ABLATION_MODE = 'full'

# RUN_MODELS = ['llama3_3', 'mistral', 'phi_medium', 'gemma_27b', 'phi_mini']
# # RUN_MODELS = ['stub']  # uncomment for testing

# # ── Timestamped output directory ──────────────────────────────────────────────
# RUN_TS   = datetime.now().strftime('%Y%m%d_%H%M%S')
# RUN_DIR  = Path('../outputs/pipeline_results') / RUN_TS
# JSON_DIR = RUN_DIR / 'json'
# INO_DIR  = RUN_DIR / 'ino'

# JSON_DIR.mkdir(parents=True, exist_ok=True)
# INO_DIR.mkdir(parents=True, exist_ok=True)
# print(f'📁 Run output directory: {RUN_DIR}')

# # ── Pre-initialise clients ONCE ───────────────────────────────────────────────
# clients: Dict[str, object] = {}
# for m in RUN_MODELS:
#     if m == 'stub':
#         clients[m] = 'stub'
#     else:
#         clients[m] = LLMClient(m)
# print(f' Clients initialised: {list(clients.keys())}')

# # ── Prompt size safety check ──────────────────────────────────────────────────
# sample_ctx = build_prompt_context(dataset[0], component_context)
# print(f'ℹ Sample prompt size: {len(sample_ctx):,} chars')
# if len(sample_ctx) > 10000:
#     logger.warning(f' Prompt is {len(sample_ctx):,} chars — risk of truncation')

# # ── Sanity test ───────────────────────────────────────────────────────────────
# print('\n── Sanity check (project 0, each model) ──')
# for m_name, m_client in clients.items():
#     test_res = run_pipeline(
#         dataset[0], component_dict, alias_to_id, component_context,
#         model=m_client, ablation='full'
#     )
#     status = 'Yes' if not test_res.get('error') else 'No'
#     print(f'  {status} {m_name}: latency={test_res["latency"]}s, '
#           f'pred={test_res["predictions"]}, err={test_res["error"]}')

# # ── Full run with incremental saves ───────────────────────────────────────────
# all_model_results: Dict[str, List[Dict]] = {}

# for model_name in RUN_MODELS:
#     print(f'\n{"=" * 60}')
#     print(f'   Running model: {model_name}')
#     print(f'{"=" * 60}')

#     client = clients[model_name]
#     model_results: List[Dict] = []
#     t0 = time.time()

#     for project in tqdm(dataset, desc=f'[{model_name}]'):
#         pid = project.get('project_id', len(model_results))
#         proj_json = JSON_DIR / f'{model_name}_P{pid:03d}.json'
        
#         if proj_json.exists():
#             with open(proj_json) as f:
#                 model_results.append(json.load(f))
#             continue
            
#         res = run_pipeline(
#             project, component_dict, alias_to_id, component_context,
#             model=client, ablation=ABLATION_MODE,
#         )
#         res['model'] = model_name

#         # ── Incremental save: per-project JSON ────────────────────────────────
#         proj_out = {k: v for k, v in res.items() if k != 'raw_output'}
#         proj_json = JSON_DIR / f'{model_name}_P{pid:03d}.json'
#         with open(proj_json, 'w') as f:
#             json.dump(proj_out, f, indent=2)

#         # ── Save .ino if generated code is present ────────────────────────────
#         raw = res.get('raw_output', {})
#         generated_code = ''
#         if isinstance(raw, dict):
#             generated_code = raw.get('code', raw.get('arduino_code', ''))
            

#         if generated_code and isinstance(generated_code, str) and generated_code.strip():
#             # Save to: <RUN_DIR>/<model_name>/code/project001.ino
#             mod_code_dir = RUN_DIR / model_name / 'code'
#             mod_code_dir.mkdir(parents=True, exist_ok=True)
#             ino_path = mod_code_dir / f'P{pid:03d}.ino'
#             with open(ino_path, 'w') as f:
#                 f.write(generated_code)

#         model_results.append(res)
#         if len(model_results) % 50 == 0:
#             print(f' {model_name}: saved {len(model_results)} projects...')

#     elapsed = time.time() - t0
#     all_model_results[model_name] = model_results

#     # ── Save full model results JSON ──────────────────────────────────────────
#     model_json_path = JSON_DIR / f'{model_name}_results.json'
#     summary_rows = []
#     for r in model_results:
#         row = {k: v for k, v in r.items() if k != 'raw_output'}
#         summary_rows.append(row)
#     with open(model_json_path, 'w') as f:
#         json.dump(summary_rows, f, indent=2)
#     print(f' Saved: {model_json_path}')

#     if isinstance(client, LLMClient):
#         ok   = sum(1 for l in client.call_log if l['success'])
#         fail = sum(1 for l in client.call_log if not l['success'])
#         print(f'{model_name}: {len(model_results)} projects in {elapsed:.1f}s '
#               f'(API: {ok} ok, {fail} failed)')
#     else:
#         print(f' {model_name}: {len(model_results)} projects in {elapsed:.1f}s')
        
#     if model_name != RUN_MODELS[-1]:
#         cooldown = 30
#         print(f'⏸ Cooling down {cooldown}s before next model...')
#         time.sleep(cooldown)

# results = all_model_results[RUN_MODELS[-1]]
# print(f'\n🏁 All models complete. Output: {RUN_DIR}')

In [ ]:
# ── CELL 13: RUN ALL PROJECTS (multi-model, incremental saves) ────────────────
from tqdm import tqdm
from datetime import datetime
import time
from openai import OpenAI

ABLATION_MODE = 'full'

RUN_MODELS = ['llama3_3', 'mistral', 'phi_medium', 'gemma_27b', 'phi_mini']
# RUN_MODELS = ['stub']  # uncomment for testing

# ── LIMIT PROJECTS (🔥 NEW) ───────────────────────────────────────────────────
MAX_PROJECTS = 3   # set to None for full dataset
dataset_subset = dataset if MAX_PROJECTS is None else dataset[:MAX_PROJECTS]

print(f'📊 Running on {len(dataset_subset)} projects')

# ── Timestamped output directory ──────────────────────────────────────────────
RUN_TS   = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR  = Path('../outputs/pipeline_results') / RUN_TS
JSON_DIR = RUN_DIR / 'json'
INO_DIR  = RUN_DIR / 'ino'

JSON_DIR.mkdir(parents=True, exist_ok=True)
INO_DIR.mkdir(parents=True, exist_ok=True)
print(f'📁 Run output directory: {RUN_DIR}')

# ── Pre-initialise clients ONCE ───────────────────────────────────────────────
clients: Dict[str, object] = {}
for m in RUN_MODELS:
    if m == 'stub':
        clients[m] = 'stub'
    else:
        clients[m] = LLMClient(m)
print(f' Clients initialised: {list(clients.keys())}')

# ── Prompt size safety check ──────────────────────────────────────────────────
sample_ctx = build_prompt_context(dataset_subset[0], component_context)
print(f'ℹ Sample prompt size: {len(sample_ctx):,} chars')
if len(sample_ctx) > 10000:
    logger.warning(f' Prompt is {len(sample_ctx):,} chars — risk of truncation')

# ── Sanity test ───────────────────────────────────────────────────────────────
print('\n── Sanity check (project 0, each model) ──')
for m_name, m_client in clients.items():
    test_res = run_pipeline(
        dataset_subset[0], component_dict, alias_to_id, component_context,
        model=m_client, ablation='full'
    )
    status = 'Yes' if not test_res.get('error') else 'No'
    print(f'  {status} {m_name}: latency={test_res["latency"]}s, '
          f'pred={test_res["predictions"]}, err={test_res["error"]}')

# ── Full run with incremental saves ───────────────────────────────────────────
all_model_results: Dict[str, List[Dict]] = {}

for model_name in RUN_MODELS:
    print(f'\n{"=" * 60}')
    print(f'   Running model: {model_name}')
    print(f'{"=" * 60}')

    client = clients[model_name]
    model_results: List[Dict] = []
    t0 = time.time()

    for project in tqdm(dataset_subset, desc=f'[{model_name}]'):  # 👈 updated
        pid = project.get('project_id', len(model_results))
        proj_json = JSON_DIR / f'{model_name}_P{pid:03d}.json'
        
        if proj_json.exists():
            with open(proj_json) as f:
                model_results.append(json.load(f))
            continue
            
        res = run_pipeline(
            project, component_dict, alias_to_id, component_context,
            model=client, ablation=ABLATION_MODE,
        )
        res['model'] = model_name

        # ── Incremental save: per-project JSON ────────────────────────────────
        proj_out = {k: v for k, v in res.items() if k != 'raw_output'}
        with open(proj_json, 'w') as f:
            json.dump(proj_out, f, indent=2)

        # ── Save .ino if generated code is present ────────────────────────────
        raw = res.get('raw_output', {})
        generated_code = ''
        if isinstance(raw, dict):
            generated_code = raw.get('code', raw.get('arduino_code', ''))

        if generated_code and isinstance(generated_code, str) and generated_code.strip():
            mod_code_dir = RUN_DIR / model_name / 'code'
            mod_code_dir.mkdir(parents=True, exist_ok=True)
            ino_path = mod_code_dir / f'P{pid:03d}.ino'
            with open(ino_path, 'w') as f:
                f.write(generated_code)

        model_results.append(res)

    elapsed = time.time() - t0
    all_model_results[model_name] = model_results

    # ── Save full model results JSON ──────────────────────────────────────────
    model_json_path = JSON_DIR / f'{model_name}_results.json'
    summary_rows = []
    for r in model_results:
        row = {k: v for k, v in r.items() if k != 'raw_output'}
        summary_rows.append(row)
    with open(model_json_path, 'w') as f:
        json.dump(summary_rows, f, indent=2)
    print(f' Saved: {model_json_path}')

    if isinstance(client, LLMClient):
        ok   = sum(1 for l in client.call_log if l['success'])
        fail = sum(1 for l in client.call_log if not l['success'])
        print(f'{model_name}: {len(model_results)} projects in {elapsed:.1f}s '
              f'(API: {ok} ok, {fail} failed)')
    else:
        print(f' {model_name}: {len(model_results)} projects in {elapsed:.1f}s')
        
    if model_name != RUN_MODELS[-1]:
        cooldown = 10  # 🔽 reduced for small runs
        print(f'⏸ Cooling down {cooldown}s before next model...')
        time.sleep(cooldown)

results = all_model_results[RUN_MODELS[-1]]
print(f'\n🏁 All models complete. Output: {RUN_DIR}')

# 🚀 Task 7: End-to-End Generation (3 Evaluation Setups)
> **Measures:** Full pipeline quality across three ablation configurations:

| Setup | Description |
|-------|-------------|
| **Full** | Description + libraries + code signals + RAG examples |
| **Desc-only** | Description only (no libraries, no code) |
| **Code-only** | Raw `.ino` code only (no description) |

**Method:** `run_pipeline(ablation='full'/'desc_only'/'code_only')` × 5 models × 434 projects. Aggregated by `aggregate_metrics` → Micro-P, Micro-R, Micro-F1, Macro-F1, latency.

**Models:** Llama 3.3, Mistral, Phi-3 Medium, Gemma 27B, Phi-3 Mini

**Status:** ❌ Not started — run Cell 60 after all infrastructure is validated.

In [ ]:
# ── CELL 14: AGGREGATION + LATENCY REPORT ─────────────────────────────────────
import numpy as np

def aggregate_metrics(results: List[Dict]) -> Dict:
    """Compute micro-averaged Precision, Recall, F1 across all projects."""
    total_tp = total_fp = total_fn = 0
    per_project_f1 = []

    for r in results:
        m = r['metrics']
        total_tp += m.get('tp', 0)
        total_fp += m.get('fp', 0)
        total_fn += m.get('fn', 0)
        per_project_f1.append(m.get('f1', 0.0))

    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1= (2 * micro_p * micro_r / (micro_p + micro_r)
               if (micro_p + micro_r) > 0 else 0.0)
    macro_f1 = sum(per_project_f1) / len(per_project_f1) if per_project_f1 else 0.0

    return {
        'n_projects'      : len(results),
        'total_tp'        : total_tp,
        'total_fp'        : total_fp,
        'total_fn'        : total_fn,
        'micro_precision' : round(micro_p,  4),
        'micro_recall'    : round(micro_r,  4),
        'micro_f1'        : round(micro_f1, 4),
        'macro_f1'        : round(macro_f1, 4),
    }


# ── Metrics table ─────────────────────────────────────────────────────────────
print('=' * 70)
print(f'{"Model":<12} {"Micro-P":>10} {"Micro-R":>10} {"Micro-F1":>10} {"Macro-F1":>10}')
print('-' * 70)

model_aggregates: Dict[str, Dict] = {}
for model_name, model_results in all_model_results.items():
    agg = aggregate_metrics(model_results)
    model_aggregates[model_name] = agg
    print(f'{model_name:<12} {agg["micro_precision"]:>10.4f} {agg["micro_recall"]:>10.4f} '
          f'{agg["micro_f1"]:>10.4f} {agg["macro_f1"]:>10.4f}')
print('=' * 70)

# ── Latency report ────────────────────────────────────────────────────────────
# NOTE: Latency is measured inside LLMClient.generate() (pure API call time).
#       Statistics are computed over successful responses only.
print(f'\n{"=" * 80}')
print(f'{"Model":<12} {"Avg (s)":>9} {"Std (s)":>9} {"P95 (s)":>9} {"Min (s)":>9} {"Max (s)":>9} {"Fail %":>9}')
print(f'{"-" * 80}')

for model_name, model_results in all_model_results.items():
    lats = [r['latency'] for r in model_results if r.get('latency', 0) > 0]
    errors = sum(1 for r in model_results if r.get('error'))  # truthy check
    fail_p = errors / len(model_results) * 100

    if lats:
        avg_l = np.mean(lats)
        std_l = np.std(lats)
        p95_l = np.percentile(lats, 95)
        min_l = np.min(lats)
        max_l = np.max(lats)
        print(f'{model_name:<12} {avg_l:>9.3f} {std_l:>9.3f} {p95_l:>9.3f} {min_l:>9.3f} {max_l:>9.3f} {fail_p:>8.1f}%')
        model_aggregates[model_name].update({
            'avg_latency_s': round(avg_l, 3),
            'std_latency_s': round(std_l, 3),
            'p95_latency_s': round(p95_l, 3),
            'error_rate_pct': round(fail_p, 1),
        })
    else:
        print(f'{model_name:<12} {"n/a":>9} {"n/a":>9} {"n/a":>9} {"n/a":>9} {"n/a":>9} {fail_p:>8.1f}%')

print(f'{"=" * 80}')

agg = model_aggregates.get(list(all_model_results.keys())[-1], {})


In [ ]:
# ── CELL 15: FAILURE ANALYSIS ─────────────────────────────────────────────────
def failure_analysis(results: List[Dict], comp_dict: Dict) -> pd.DataFrame:
    """Compute per-project FP/FN counts and most common error components."""
    rows = []
    fn_counter: Dict[int, int] = defaultdict(int)
    fp_counter: Dict[int, int] = defaultdict(int)

    for r in results:
        pred = set(r['predictions'])
        gt   = set(r['ground_truth'])
        fn_ids = gt  - pred
        fp_ids = pred - gt
        for cid in fn_ids:
            fn_counter[cid] += 1
        for cid in fp_ids:
            fp_counter[cid] += 1
        rows.append({
            'project_id': r['project_id'],
            'n_missing' : len(fn_ids),
            'n_extra'   : len(fp_ids),
            'missing_ids': sorted(fn_ids),
            'extra_ids'  : sorted(fp_ids),
        })

    df = pd.DataFrame(rows)
    avg_fn = df['n_missing'].mean()
    avg_fp = df['n_extra'].mean()
    print(f'Avg missing components (FN): {avg_fn:.2f}')
    print(f'Avg extra   components (FP): {avg_fp:.2f}')

    print('\nTop 10 most-missed components (FN):')
    for cid, cnt in sorted(fn_counter.items(), key=lambda x: -x[1])[:10]:
        name = comp_dict.get(cid, {}).get('name', '?')
        print(f'  [{cid}] {name}: missed {cnt}x')

    print('\nTop 10 most-hallucinated components (FP):')
    for cid, cnt in sorted(fp_counter.items(), key=lambda x: -x[1])[:10]:
        name = comp_dict.get(cid, {}).get('name', '?')
        print(f'  [{cid}] {name}: hallucinated {cnt}x')

    return df


failure_df = failure_analysis(results, component_dict)
failure_df.head(10)

In [ ]:
# ── CELL 16: DEBUG LOGGING ────────────────────────────────────────────────────
def debug_log(results: List[Dict]) -> None:
    """Print a structured debug report for problematic predictions."""
    empty_outputs    = [r for r in results if not r['predictions'] and not r['ground_truth']]
    json_failures    = [r for r in results if r['error'] is not None]
    bad_predictions  = [r for r in results if r['metrics'].get('f1', 1.0) == 0.0
                        and r['ground_truth']]
    hallucinations   = [r for r in results
                        if r['metrics'].get('fp', 0) > 0 and r['metrics'].get('tp', 0) == 0]

    sep = '─' * 55
    print(sep)
    print(f'DEBUG REPORT  |  {len(results)} projects')
    print(sep)
    print(f'  Empty outputs (both pred & GT empty)  : {len(empty_outputs)}')
    print(f'  JSON / pipeline failures               : {len(json_failures)}')
    print(f'  Zero-F1 predictions (GT non-empty)     : {len(bad_predictions)}')
    print(f'  Pure hallucination (TP=0, FP>0)        : {len(hallucinations)}')
    print(sep)

    if json_failures:
        print('\nJSON FAILURES:')
        for r in json_failures[:5]:
            print(f'  project {r["project_id"]}: {r["error"]}')

    if bad_predictions:
        print('\nZERO-F1 SAMPLES:')
        for r in bad_predictions[:5]:
            print(f'  project {r["project_id"]} | pred={r["predictions"]} | gt={r["ground_truth"]}')


debug_log(results)

In [ ]:
# ── CELL 17: ABLATION STUDY ───────────────────────────────────────────────────
ABLATION_MODES = ['full', 'desc_only', 'code_only']
ablation_summary: List[Dict] = []

for mode in ABLATION_MODES:
    print(f'\n⏳ Running ablation: {mode} ...')
    mode_results: List[Dict] = []
    for project in dataset:
        res = run_pipeline(
            project,
            component_dict,
            alias_to_id,
            component_context,
            model=ACTIVE_MODEL,
            ablation=mode,
        )
        mode_results.append(res)

    agg_mode = aggregate_metrics(mode_results)
    agg_mode['ablation'] = mode
    ablation_summary.append(agg_mode)
    print(f'  micro_f1={agg_mode["micro_f1"]:.4f}  macro_f1={agg_mode["macro_f1"]:.4f}')


ablation_df = pd.DataFrame(ablation_summary)[['ablation', 'micro_precision',
                                               'micro_recall', 'micro_f1', 'macro_f1']]
print('\n=== ABLATION COMPARISON ===')
print(ablation_df.to_string(index=False))

In [ ]:
# ── CELL 18: EXPORT RESULTS (into run directory) ──────────────────────────────

# ── Per-project CSV (all models combined) ─────────────────────────────────────
rows = []
for model_name, model_results in all_model_results.items():
    for r in model_results:
        row = {'model': model_name, 'project_id': r['project_id'], 'ablation': r['ablation'],
               'latency': r.get('latency', 0.0)}
        row.update(r['metrics'])
        row['predictions']  = json.dumps(r['predictions'])
        row['ground_truth'] = json.dumps(r['ground_truth'])
        row['error']        = r['error']
        rows.append(row)

results_df = pd.DataFrame(rows)
results_csv = RUN_DIR / 'llm_benchmark_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'Per-project results → {results_csv}')

# ── Aggregate JSON ────────────────────────────────────────────────────────────
agg_json = RUN_DIR / 'llm_benchmark_aggregate.json'
with open(agg_json, 'w') as f:
    json.dump(model_aggregates, f, indent=2)
print(f' Aggregated metrics  → {agg_json}')

# ── Model comparison CSV ──────────────────────────────────────────────────────
summary_df = pd.DataFrame(model_aggregates).T
summary_csv = RUN_DIR / 'model_summary.csv'
summary_df.to_csv(summary_csv)
print(f' Model summary       → {summary_csv}')

# ── Also copy to top-level outputs/ for easy access ───────────────────────────
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
results_df.to_csv(OUTPUT_DIR / 'llm_benchmark_results.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'model_summary.csv')
with open(OUTPUT_DIR / 'llm_benchmark_aggregate.json', 'w') as f:
    json.dump(model_aggregates, f, indent=2)
print(f' Latest copies       → {OUTPUT_DIR}/')

print(f'\n Full run archived at: {RUN_DIR}')
print(summary_df.to_string())
